In [ ]:
# ============================================================
# STEP 1: Corpus Collection
# CS + Medical | AI vs Human
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings("ignore")

# ============================================================
# CHANGE THESE TWO PATHS ACCORDING TO YOUR GOOGLE DRIVE
# ============================================================

CS_FILE = "/content/drive/MyDrive/cs_domain_main_cs_merged_abstracts.csv"
MED_FILE = "/content/drive/MyDrive/medi_domain_main_file_merged_abstracts_medi.csv"

OUTPUT_DIR = "/content/drive/MyDrive/cross_domain_scibert_tfidf_umap_stepwise"
os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_STATE = 42

# ============================================================
# Safe CSV reader
# ============================================================

def read_csv_safely(path):
    encodings = ["utf-8", "utf-8-sig", "cp1252", "latin1"]

    for enc in encodings:
        try:
            df = pd.read_csv(path, encoding=enc)
            print(f"Loaded: {os.path.basename(path)} using encoding: {enc}")
            return df
        except Exception as e:
            print(f"Failed with encoding {enc}: {e}")

    raise RuntimeError(f"Could not read file: {path}")

cs_df = read_csv_safely(CS_FILE)
med_df = read_csv_safely(MED_FILE)

# Remove unnecessary unnamed columns
cs_df = cs_df.drop(columns=[c for c in cs_df.columns if str(c).startswith("Unnamed")], errors="ignore")
med_df = med_df.drop(columns=[c for c in med_df.columns if str(c).startswith("Unnamed")], errors="ignore")

print("\n========== RAW DATASET INFORMATION ==========")
print("CS shape:", cs_df.shape)
print("CS columns:", cs_df.columns.tolist())

print("\nMedical shape:", med_df.shape)
print("Medical columns:", med_df.columns.tolist())

# ============================================================
# Column settings
# If your column names are different, change them here.
# ============================================================

CS_TITLE_COL = "title"
CS_HUMAN_COL = "original_abstract"
CS_AI_COL = "ai_generated_abstract"

MED_TITLE_COL = "title"
MED_HUMAN_COL = "abstract"
MED_AI_COL = "ai_generated_abstract"

required_cs_cols = [CS_TITLE_COL, CS_HUMAN_COL, CS_AI_COL]
required_med_cols = [MED_TITLE_COL, MED_HUMAN_COL, MED_AI_COL]

for col in required_cs_cols:
    if col not in cs_df.columns:
        raise ValueError(f"CS file missing required column: {col}")

for col in required_med_cols:
    if col not in med_df.columns:
        raise ValueError(f"Medical file missing required column: {col}")

# Convert required columns to string
for col in required_cs_cols:
    cs_df[col] = cs_df[col].fillna("").astype(str)

for col in required_med_cols:
    med_df[col] = med_df[col].fillna("").astype(str)

# ============================================================
# Convert CS and Medical datasets into long format
# Required format:
# domain | label | title | text_raw
# ============================================================

cs_human = pd.DataFrame({
    "domain": "CS",
    "label": "Human",
    "title": cs_df[CS_TITLE_COL],
    "text_raw": cs_df[CS_HUMAN_COL],
    "source_column": CS_HUMAN_COL
})

cs_ai = pd.DataFrame({
    "domain": "CS",
    "label": "AI",
    "title": cs_df[CS_TITLE_COL],
    "text_raw": cs_df[CS_AI_COL],
    "source_column": CS_AI_COL
})

med_human = pd.DataFrame({
    "domain": "Medical",
    "label": "Human",
    "title": med_df[MED_TITLE_COL],
    "text_raw": med_df[MED_HUMAN_COL],
    "source_column": MED_HUMAN_COL
})

med_ai = pd.DataFrame({
    "domain": "Medical",
    "label": "AI",
    "title": med_df[MED_TITLE_COL],
    "text_raw": med_df[MED_AI_COL],
    "source_column": MED_AI_COL
})

corpus_df = pd.concat(
    [cs_human, cs_ai, med_human, med_ai],
    ignore_index=True
)

corpus_df["doc_id"] = [f"DOC_{i:05d}" for i in range(len(corpus_df))]

corpus_df = corpus_df[
    ["doc_id", "domain", "label", "title", "source_column", "text_raw"]
]

# Basic raw-text check
corpus_df["raw_word_count"] = corpus_df["text_raw"].apply(
    lambda x: len(re.findall(r"\b\w+\b", str(x)))
)

print("\n========== STEP 1 RESULT: CORPUS COLLECTION ==========")
print("Long-format corpus shape:", corpus_df.shape)

print("\nDocuments by domain and label:")
display(corpus_df.groupby(["domain", "label"]).size().reset_index(name="documents"))

print("\nRaw word-count summary:")
display(
    corpus_df.groupby(["domain", "label"])["raw_word_count"]
    .describe()
    .round(2)
)

print("\nSample rows:")
display(corpus_df.head())

# Save Step 1 output
step1_output_path = os.path.join(OUTPUT_DIR, "step1_long_format_corpus_raw.csv")
corpus_df.to_csv(step1_output_path, index=False, encoding="utf-8-sig")

print("\nSaved Step 1 output to:")
print(step1_output_path)

In [ ]:
# ============================================================
# STEP 2: Preprocessing and Quality Audit
# Cleaning | Pair filtering | Tokenization | Lemmatization
# ============================================================

!pip install -q ftfy beautifulsoup4 nltk

import os
import re
import html
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from bs4 import BeautifulSoup
import ftfy

tqdm.pandas()

import nltk
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# ============================================================
# 2.1 Basic cleaning functions
# ============================================================

def safe_str(x):
    if pd.isna(x):
        return ""
    return str(x)

def normalize_whitespace(text):
    text = safe_str(text)
    text = text.replace("\r", " ").replace("\n", " ").replace("\t", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def fix_encoding(text):
    text = safe_str(text)
    text = ftfy.fix_text(text)

    replacements = {
        "Ã¢ÂÂ": "'",
        "Ã¢ÂÂ": "'",
        "Ã¢ÂÂ": '"',
        "Ã¢ÂÂ": '"',
        "Ã¢ÂÂ": "-",
        "Ã¢ÂÂ": "-",
        "Ã¢ÂÂ¦": "...",
        "â€™": "'",
        "â€˜": "'",
        "â€œ": '"',
        "â€": '"',
        "â€“": "-",
        "â€”": "-",
        "â€¦": "...",
        "Â": "",
        "\x81": "",
    }

    for bad, good in replacements.items():
        text = text.replace(bad, good)

    return text

def remove_html(text):
    text = safe_str(text)
    text = html.unescape(text)
    text = re.sub(r"<[^>]*>", " ", text)
    text = BeautifulSoup(text, "html.parser").get_text(" ")
    return normalize_whitespace(text)

def remove_prompt_echo(text):
    text = safe_str(text)
    text = normalize_whitespace(text)

    leading_patterns = [
        r"^\s*here is .*?abstract[:\-]?\s*",
        r"^\s*here are .*?abstracts[:\-]?\s*",
        r"^\s*below is .*?abstract[:\-]?\s*",
        r"^\s*the following is .*?abstract[:\-]?\s*",
        r"^\s*this is .*?abstract[:\-]?\s*",
        r"^\s*rewritten version[:\-]?\s*",
        r"^\s*rewritten abstract[:\-]?\s*",
        r"^\s*new ai generated abstract[:\-]?\s*",
        r"^\s*ai generated abstract[:\-]?\s*",
        r"^\s*generated abstract[:\-]?\s*",
        r"^\s*certainly[,.]?\s*",
        r"^\s*sure[,.]?\s*",
        r"^\s*of course[,.]?\s*",
    ]

    for pat in leading_patterns:
        text = re.sub(pat, "", text, flags=re.IGNORECASE)

    label_patterns = [
        r"\bNew AI Generated Abstract\s*:\s*",
        r"\bAI Generated Abstract\s*:\s*",
        r"\bGenerated Abstract\s*:\s*",
        r"\bRewritten Abstract\s*:\s*",
        r"\bOriginal Abstract\s*:\s*",
        r"\bHuman Abstract\s*:\s*",
        r"\bTitle\s*:\s*",
        r"\bAbstract\s*:\s*",
    ]

    for pat in label_patterns:
        text = re.sub(pat, "", text, flags=re.IGNORECASE)

    phrase_patterns = [
        r"\bhere is a rewritten version\b",
        r"\bhere is the rewritten version\b",
        r"\bhere is the abstract\b",
        r"\bhere is an abstract\b",
        r"\bthe rewritten abstract is\b",
        r"\bthis rewritten abstract\b",
        r"\bas an ai language model\b",
    ]

    for pat in phrase_patterns:
        text = re.sub(pat, "", text, flags=re.IGNORECASE)

    return normalize_whitespace(text)

def clean_text_basic(text, label):
    text = safe_str(text)
    text = fix_encoding(text)
    text = remove_html(text)

    if label == "AI":
        text = remove_prompt_echo(text)

    text = normalize_whitespace(text)
    return text

def word_count(text):
    return len(re.findall(r"\b\w+\b", safe_str(text)))

def contains_html(text):
    return bool(re.search(r"<[^>]+>", safe_str(text)))

def contains_encoding_artifact(text):
    bad_patterns = ["Ã", "Â", "â€", "Ã¢", "\x81"]
    text = safe_str(text)
    return any(p in text for p in bad_patterns)

def contains_prompt_echo(text):
    low = safe_str(text).lower()
    patterns = [
        "here is the abstract",
        "here is a rewritten",
        "rewritten abstract:",
        "new ai generated abstract:",
        "ai generated abstract:",
        "original abstract:",
        "generated abstract:",
        "as an ai language model",
        "chatgpt"
    ]
    return any(p in low for p in patterns)

# ============================================================
# 2.2 Apply cleaning
# ============================================================

corpus_df["text_clean"] = corpus_df.progress_apply(
    lambda row: clean_text_basic(row["text_raw"], row["label"]),
    axis=1
)

corpus_df["clean_word_count"] = corpus_df["text_clean"].apply(word_count)

# ============================================================
# 2.3 Create pair IDs and remove invalid empty pairs
# Important: if one side of a Medical pair is empty, remove both AI and Human
# ============================================================

corpus_df["source_row_id"] = corpus_df.groupby(["domain", "label"]).cumcount()
corpus_df["pair_id"] = corpus_df["domain"] + "_" + corpus_df["source_row_id"].astype(str)

corpus_df["valid_clean_text"] = corpus_df["text_clean"].str.strip() != ""

pair_validity = (
    corpus_df
    .groupby("pair_id")
    .agg(
        domain=("domain", "first"),
        labels_present=("label", "nunique"),
        pair_docs=("doc_id", "count"),
        both_text_valid=("valid_clean_text", "min")
    )
    .reset_index()
)

valid_pair_ids = pair_validity[
    (pair_validity["labels_present"] == 2) &
    (pair_validity["pair_docs"] == 2) &
    (pair_validity["both_text_valid"] == True)
]["pair_id"]

before_pair_filter = len(corpus_df)
corpus_df = corpus_df[corpus_df["pair_id"].isin(valid_pair_ids)].copy().reset_index(drop=True)
after_pair_filter = len(corpus_df)

# Recreate document IDs after filtering
corpus_df["doc_id"] = [f"DOC_{i:05d}" for i in range(len(corpus_df))]

print("========== STEP 2A CLEANING + PAIR FILTERING ==========")
print("Documents before pair filtering:", before_pair_filter)
print("Documents after pair filtering:", after_pair_filter)
print("Documents removed:", before_pair_filter - after_pair_filter)

print("\nDocuments by domain and label after cleaning/filtering:")
display(corpus_df.groupby(["domain", "label"]).size().reset_index(name="documents"))

# ============================================================
# 2.4 Leakage / prompt echo check
# ============================================================

leak_patterns = [
    "ai generated abstract",
    "new ai generated abstract",
    "generated abstract",
    "human abstract",
    "original abstract",
    "chatgpt",
    "as an ai language model",
    "here is the abstract",
    "rewritten abstract"
]

print("\n========== LEAKAGE / PROMPT ECHO CHECK ==========")

leak_summary = []

for pat in leak_patterns:
    count = corpus_df["text_clean"].str.lower().str.contains(pat, regex=False, na=False).sum()
    leak_summary.append({"pattern": pat, "remaining_count": int(count)})
    print(f"{pat}: {count}")

leak_summary_df = pd.DataFrame(leak_summary)

# ============================================================
# 2.5 Tokenization and lemmatization for TF-IDF
# SciBERT will later use text_clean directly
# ============================================================

stop_words = set(stopwords.words("english"))

# Keep useful contrast words that may matter in writing style
keep_words = {
    "not", "no", "nor",
    "against",
    "between",
    "under",
    "over",
    "more",
    "most"
}

stop_words = stop_words - keep_words
lemmatizer = WordNetLemmatizer()

def preprocess_for_tfidf(text):
    text = safe_str(text).lower()

    # Regex tokenizer avoids punkt errors
    tokens = re.findall(r"\b[a-zA-Z]+\b", text)

    processed = []

    for tok in tokens:
        tok = tok.lower().strip()

        if len(tok) < 2:
            continue

        if tok in stop_words:
            continue

        lemma = lemmatizer.lemmatize(tok)

        if len(lemma) < 2:
            continue

        processed.append(lemma)

    return processed

corpus_df["tokens"] = corpus_df["text_clean"].progress_apply(preprocess_for_tfidf)
corpus_df["preprocessed_text"] = corpus_df["tokens"].apply(lambda toks: " ".join(toks))
corpus_df["token_count"] = corpus_df["tokens"].apply(len)

# Remove any pair where tokenized text became empty
corpus_df["valid_preprocessed_text"] = corpus_df["preprocessed_text"].str.strip() != ""

pair_validity_after_tokens = (
    corpus_df
    .groupby("pair_id")
    .agg(
        labels_present=("label", "nunique"),
        pair_docs=("doc_id", "count"),
        both_tokens_valid=("valid_preprocessed_text", "min")
    )
    .reset_index()
)

valid_pair_ids_after_tokens = pair_validity_after_tokens[
    (pair_validity_after_tokens["labels_present"] == 2) &
    (pair_validity_after_tokens["pair_docs"] == 2) &
    (pair_validity_after_tokens["both_tokens_valid"] == True)
]["pair_id"]

before_token_filter = len(corpus_df)
corpus_df = corpus_df[corpus_df["pair_id"].isin(valid_pair_ids_after_tokens)].copy().reset_index(drop=True)
after_token_filter = len(corpus_df)

corpus_df["doc_id"] = [f"DOC_{i:05d}" for i in range(len(corpus_df))]

print("\n========== STEP 2B TOKENIZATION + LEMMATIZATION ==========")
print("Documents before token-empty filtering:", before_token_filter)
print("Documents after token-empty filtering:", after_token_filter)
print("Documents removed after tokenization:", before_token_filter - after_token_filter)

token_audit = corpus_df.groupby(["domain", "label"]).agg(
    docs=("doc_id", "count"),
    mean_tokens=("token_count", "mean"),
    min_tokens=("token_count", "min"),
    max_tokens=("token_count", "max"),
).reset_index()

display(token_audit)

# ============================================================
# 2.6 Final quality audit
# ============================================================

quality_counts = {
    "total_documents": len(corpus_df),
    "empty_clean_text": int((corpus_df["text_clean"].str.strip() == "").sum()),
    "html_remaining": int(corpus_df["text_clean"].apply(contains_html).sum()),
    "encoding_artifacts_remaining": int(corpus_df["text_clean"].apply(contains_encoding_artifact).sum()),
    "prompt_echo_remaining_in_AI": int(
        corpus_df.loc[corpus_df["label"] == "AI", "text_clean"]
        .apply(contains_prompt_echo)
        .sum()
    ),
    "empty_preprocessed_documents": int((corpus_df["preprocessed_text"].str.strip() == "").sum()),
    "human_under_50_words_kept": int(
        ((corpus_df["label"] == "Human") & (corpus_df["clean_word_count"] < 50)).sum()
    ),
    "ai_under_100_words": int(
        ((corpus_df["label"] == "AI") & (corpus_df["clean_word_count"] < 100)).sum()
    ),
}

quality_df = pd.DataFrame(
    list(quality_counts.items()),
    columns=["quality_item", "final_value"]
)

print("\n========== FINAL STEP 2 QUALITY AUDIT ==========")
display(quality_df)

print("\nClean word-count summary:")
display(
    corpus_df.groupby(["domain", "label"])["clean_word_count"]
    .describe()
    .round(2)
)

print("\nFinal documents by domain and label:")
display(corpus_df.groupby(["domain", "label"]).size().reset_index(name="documents"))

print("\nSample cleaned rows:")
display(
    corpus_df[
        ["doc_id", "domain", "label", "title", "clean_word_count", "token_count", "text_clean", "preprocessed_text"]
    ].head()
)

# ============================================================
# 2.7 Save Step 2 outputs
# ============================================================

step2_output_path = os.path.join(OUTPUT_DIR, "step2_cleaned_preprocessed_corpus.csv")
quality_output_path = os.path.join(OUTPUT_DIR, "step2_quality_audit.csv")
token_audit_output_path = os.path.join(OUTPUT_DIR, "step2_token_audit.csv")
leak_output_path = os.path.join(OUTPUT_DIR, "step2_leakage_check.csv")

corpus_df.to_csv(step2_output_path, index=False, encoding="utf-8-sig")
quality_df.to_csv(quality_output_path, index=False, encoding="utf-8-sig")
token_audit.to_csv(token_audit_output_path, index=False, encoding="utf-8-sig")
leak_summary_df.to_csv(leak_output_path, index=False, encoding="utf-8-sig")

print("\nSaved Step 2 outputs:")
print(step2_output_path)
print(quality_output_path)
print(token_audit_output_path)
print(leak_output_path)

In [ ]:
# ============================================================
# STEP 2.1: Fix remaining prompt echo / label artifacts
# Run this after Step 2 and before Step 3
# ============================================================

print("========== BEFORE FIX: AI ROWS WITH POSSIBLE PROMPT ECHO ==========")

ai_leak_patterns = [
    "ai generated abstract",
    "new ai generated abstract",
    "generated abstract",
    "rewritten abstract",
    "original abstract",
    "here is the abstract",
    "as an ai language model"
]

for pat in ai_leak_patterns:
    matched = corpus_df[
        (corpus_df["label"] == "AI") &
        (corpus_df["text_clean"].str.lower().str.contains(pat, regex=False, na=False))
    ]
    print(f"\nPattern: {pat} | AI matches: {len(matched)}")
    if len(matched) > 0:
        display(matched[["doc_id", "domain", "label", "title", "text_clean"]].head(5))


def remove_residual_prompt_artifacts(text):
    text = safe_str(text)

    residual_patterns = [
        r"\bnew\s+ai\s+generated\s+abstract\s*:?\s*",
        r"\bai\s+generated\s+abstract\s*:?\s*",
        r"\bgenerated\s+abstract\s*:?\s*",
        r"\brewritten\s+abstract\s*:?\s*",
        r"\boriginal\s+abstract\s*:?\s*",
        r"\bhuman\s+abstract\s*:?\s*",
        r"\bhere\s+is\s+the\s+abstract\s*:?\s*",
        r"\bhere\s+is\s+a\s+rewritten\s+version\s*:?\s*",
        r"\bhere\s+is\s+the\s+rewritten\s+version\s*:?\s*",
        r"\bas\s+an\s+ai\s+language\s+model\s*:?\s*",
    ]

    for pat in residual_patterns:
        text = re.sub(pat, " ", text, flags=re.IGNORECASE)

    text = normalize_whitespace(text)
    return text


# Apply extra cleanup only to AI rows
ai_idx = corpus_df["label"] == "AI"
corpus_df.loc[ai_idx, "text_clean"] = corpus_df.loc[ai_idx, "text_clean"].apply(
    remove_residual_prompt_artifacts
)

# Recalculate clean word count
corpus_df["clean_word_count"] = corpus_df["text_clean"].apply(word_count)

# Rebuild TF-IDF tokens after cleanup
corpus_df["tokens"] = corpus_df["text_clean"].progress_apply(preprocess_for_tfidf)
corpus_df["preprocessed_text"] = corpus_df["tokens"].apply(lambda toks: " ".join(toks))
corpus_df["token_count"] = corpus_df["tokens"].apply(len)

# Check if anything became empty
empty_clean = int((corpus_df["text_clean"].str.strip() == "").sum())
empty_preprocessed = int((corpus_df["preprocessed_text"].str.strip() == "").sum())

print("\n========== AFTER FIX: LEAKAGE / PROMPT ECHO CHECK ==========")

leak_patterns = [
    "ai generated abstract",
    "new ai generated abstract",
    "generated abstract",
    "human abstract",
    "original abstract",
    "chatgpt",
    "as an ai language model",
    "here is the abstract",
    "rewritten abstract"
]

leak_summary_after = []

for pat in leak_patterns:
    total_count = corpus_df["text_clean"].str.lower().str.contains(pat, regex=False, na=False).sum()
    ai_count = corpus_df[
        (corpus_df["label"] == "AI") &
        (corpus_df["text_clean"].str.lower().str.contains(pat, regex=False, na=False))
    ].shape[0]

    leak_summary_after.append({
        "pattern": pat,
        "total_remaining_count": int(total_count),
        "ai_remaining_count": int(ai_count)
    })

leak_summary_after_df = pd.DataFrame(leak_summary_after)
display(leak_summary_after_df)

print("\nEmpty clean text:", empty_clean)
print("Empty preprocessed text:", empty_preprocessed)

print("\n========== UPDATED TOKEN AUDIT ==========")
token_audit_updated = corpus_df.groupby(["domain", "label"]).agg(
    docs=("doc_id", "count"),
    mean_tokens=("token_count", "mean"),
    min_tokens=("token_count", "min"),
    max_tokens=("token_count", "max"),
).reset_index()

display(token_audit_updated)

print("\n========== UPDATED FINAL DOCUMENT COUNTS ==========")
display(corpus_df.groupby(["domain", "label"]).size().reset_index(name="documents"))

# Save corrected Step 2 output
step2_corrected_output_path = os.path.join(
    OUTPUT_DIR,
    "step2_cleaned_preprocessed_corpus_corrected.csv"
)

corpus_df.to_csv(step2_corrected_output_path, index=False, encoding="utf-8-sig")

print("\nSaved corrected Step 2 output to:")
print(step2_corrected_output_path)

In [ ]:
# ============================================================
# STEP 3: Domain A / CS-only feature learning
# TF-IDF fitted only on CS
# SciBERT embeddings extracted only for CS
# ============================================================

!pip install -q scikit-learn transformers accelerate sentencepiece torch joblib

import os
import numpy as np
import pandas as pd
import joblib
from tqdm.auto import tqdm

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler

import torch
from transformers import AutoTokenizer, AutoModel

# ============================================================
# 3.1 Create output folders
# ============================================================

STEP3_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "step3_domain_a_feature_learning")
TFIDF_OUTPUT_DIR = os.path.join(STEP3_OUTPUT_DIR, "tfidf")
SCIBERT_OUTPUT_DIR = os.path.join(STEP3_OUTPUT_DIR, "scibert")

os.makedirs(STEP3_OUTPUT_DIR, exist_ok=True)
os.makedirs(TFIDF_OUTPUT_DIR, exist_ok=True)
os.makedirs(SCIBERT_OUTPUT_DIR, exist_ok=True)

# ============================================================
# 3.2 Select Domain A = CS only
# Medical must NOT be used in Step 3
# ============================================================

cs_docs = corpus_df[
    (corpus_df["domain"] == "CS") &
    (corpus_df["text_clean"].str.strip() != "") &
    (corpus_df["preprocessed_text"].str.strip() != "")
].copy().reset_index(drop=True)

medical_docs_check = cs_docs[cs_docs["domain"] == "Medical"]

print("========== STEP 3A: DOMAIN A SELECTION ==========")
print("CS-only documents:", len(cs_docs))
print("Medical documents used in Step 3:", len(medical_docs_check))

print("\nCS label distribution:")
display(cs_docs["label"].value_counts().reset_index().rename(columns={"index": "label", "label": "documents"}))

# Masks for later contrast computation
cs_ai_mask = cs_docs["label"].values == "AI"
cs_human_mask = cs_docs["label"].values == "Human"

print("\nCS AI documents:", int(cs_ai_mask.sum()))
print("CS Human documents:", int(cs_human_mask.sum()))

# Save CS-only dataframe
cs_docs_path = os.path.join(STEP3_OUTPUT_DIR, "step3_domain_a_cs_only_documents.csv")
cs_docs.to_csv(cs_docs_path, index=False, encoding="utf-8-sig")

print("\nSaved CS-only document file:")
print(cs_docs_path)

# ============================================================
# 3.3 TF-IDF fitted on CS only
# This follows the original pipeline rule:
# fit on Domain A only, not Medical
# ============================================================

TFIDF_CONFIG = {
    "ngram_range": (1, 2),
    "min_df": 5,
    "max_df": 0.85,
    "max_features": 30000,
    "sublinear_tf": True,
    "norm": "l2",
    "lowercase": False,
    "dtype": np.float32
}

tfidf_vectorizer_cs = TfidfVectorizer(**TFIDF_CONFIG)

X_cs_tfidf = tfidf_vectorizer_cs.fit_transform(cs_docs["preprocessed_text"])

tfidf_feature_names = np.array(tfidf_vectorizer_cs.get_feature_names_out())

tfidf_features_df = pd.DataFrame({
    "feature_id": np.arange(len(tfidf_feature_names)),
    "feature": tfidf_feature_names,
    "idf": tfidf_vectorizer_cs.idf_
})

tfidf_features_df["ngram_type"] = tfidf_features_df["feature"].apply(
    lambda x: "bigram" if " " in x else "unigram"
)

print("\n========== STEP 3B: TF-IDF FITTED ON CS ONLY ==========")
print("TF-IDF matrix shape:", X_cs_tfidf.shape)
print("Number of CS documents:", X_cs_tfidf.shape[0])
print("Number of TF-IDF features:", X_cs_tfidf.shape[1])
print("Non-zero values:", X_cs_tfidf.nnz)
print("Sparsity:", round(1 - (X_cs_tfidf.nnz / (X_cs_tfidf.shape[0] * X_cs_tfidf.shape[1])), 6))
print("All-zero rows:", int((np.diff(X_cs_tfidf.indptr) == 0).sum()))

print("\nSample TF-IDF features:")
display(tfidf_features_df.head(10))

# Save TF-IDF objects
tfidf_vectorizer_path = os.path.join(TFIDF_OUTPUT_DIR, "step3_tfidf_vectorizer_fitted_on_cs_only.joblib")
tfidf_matrix_path = os.path.join(TFIDF_OUTPUT_DIR, "step3_X_cs_tfidf.npz")
tfidf_features_path = os.path.join(TFIDF_OUTPUT_DIR, "step3_tfidf_features.csv")

joblib.dump(tfidf_vectorizer_cs, tfidf_vectorizer_path)

from scipy import sparse
sparse.save_npz(tfidf_matrix_path, X_cs_tfidf)

tfidf_features_df.to_csv(tfidf_features_path, index=False, encoding="utf-8-sig")

print("\nSaved TF-IDF outputs:")
print(tfidf_vectorizer_path)
print(tfidf_matrix_path)
print(tfidf_features_path)

# ============================================================
# 3.4 SciBERT embeddings for CS only
# SciBERT uses cleaned natural text, not lemmatized TF-IDF text
# ============================================================

SCIBERT_MODEL_NAME = "allenai/scibert_scivocab_uncased"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("\n========== STEP 3C: SCIBERT SETUP ==========")
print("Using device:", device)

if str(device) == "cpu":
    print("WARNING: You are using CPU. SciBERT will be slow. Runtime > Change runtime type > GPU is recommended.")

tokenizer = AutoTokenizer.from_pretrained(SCIBERT_MODEL_NAME)
scibert_model = AutoModel.from_pretrained(SCIBERT_MODEL_NAME).to(device)
scibert_model.eval()

def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts

def embed_texts_scibert(texts, batch_size=16, max_length=512):
    all_embeddings = []
    texts = list(texts)

    for start in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[start:start + batch_size]

        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        encoded = {k: v.to(device) for k, v in encoded.items()}

        with torch.no_grad():
            outputs = scibert_model(**encoded)
            embeddings = mean_pooling(outputs.last_hidden_state, encoded["attention_mask"])

        all_embeddings.append(embeddings.cpu().numpy())

    return np.vstack(all_embeddings).astype(np.float32)

cs_scibert_raw_path = os.path.join(SCIBERT_OUTPUT_DIR, "step3_cs_scibert_raw_embeddings.npy")

if os.path.exists(cs_scibert_raw_path):
    E_cs_scibert_raw = np.load(cs_scibert_raw_path)
    print("\nLoaded existing CS SciBERT embeddings:")
    print(cs_scibert_raw_path)
else:
    E_cs_scibert_raw = embed_texts_scibert(
        cs_docs["text_clean"].fillna("").astype(str).values,
        batch_size=16,
        max_length=512
    )
    np.save(cs_scibert_raw_path, E_cs_scibert_raw)
    print("\nSaved new CS SciBERT embeddings:")
    print(cs_scibert_raw_path)

print("\n========== STEP 3D: SCIBERT EMBEDDINGS COMPLETE ==========")
print("CS SciBERT raw embedding shape:", E_cs_scibert_raw.shape)

# ============================================================
# 3.5 Standardize SciBERT embeddings using CS only
# This scaler will later be applied to Medical
# ============================================================

scibert_scaler_cs = StandardScaler()
E_cs_scibert = scibert_scaler_cs.fit_transform(E_cs_scibert_raw).astype(np.float32)

cs_scibert_scaled_path = os.path.join(SCIBERT_OUTPUT_DIR, "step3_cs_scibert_scaled_embeddings.npy")
scibert_scaler_path = os.path.join(SCIBERT_OUTPUT_DIR, "step3_scibert_scaler_fitted_on_cs_only.joblib")

np.save(cs_scibert_scaled_path, E_cs_scibert)
joblib.dump(scibert_scaler_cs, scibert_scaler_path)

print("\nSciBERT scaled embedding shape:", E_cs_scibert.shape)
print("SciBERT embedding dimensions:", E_cs_scibert.shape[1])

print("\nSaved SciBERT outputs:")
print(cs_scibert_raw_path)
print(cs_scibert_scaled_path)
print(scibert_scaler_path)

# ============================================================
# 3.6 Final Step 3 summary
# ============================================================

step3_summary = pd.DataFrame([
    {
        "item": "CS-only documents",
        "value": len(cs_docs)
    },
    {
        "item": "CS AI documents",
        "value": int(cs_ai_mask.sum())
    },
    {
        "item": "CS Human documents",
        "value": int(cs_human_mask.sum())
    },
    {
        "item": "Medical documents used in Step 3",
        "value": len(medical_docs_check)
    },
    {
        "item": "TF-IDF matrix shape",
        "value": str(X_cs_tfidf.shape)
    },
    {
        "item": "TF-IDF features",
        "value": X_cs_tfidf.shape[1]
    },
    {
        "item": "TF-IDF all-zero rows",
        "value": int((np.diff(X_cs_tfidf.indptr) == 0).sum())
    },
    {
        "item": "SciBERT raw embedding shape",
        "value": str(E_cs_scibert_raw.shape)
    },
    {
        "item": "SciBERT scaled embedding shape",
        "value": str(E_cs_scibert.shape)
    },
    {
        "item": "SciBERT embedding dimensions",
        "value": E_cs_scibert.shape[1]
    },
])

print("\n========== FINAL STEP 3 SUMMARY ==========")
display(step3_summary)

step3_summary_path = os.path.join(STEP3_OUTPUT_DIR, "step3_summary.csv")
step3_summary.to_csv(step3_summary_path, index=False, encoding="utf-8-sig")

print("\nSaved Step 3 summary:")
print(step3_summary_path)

In [ ]:
# ============================================================
# STEP 4: Contrast Computation
# TF-IDF_AI - TF-IDF_Human
# SciBERT_AI - SciBERT_Human
# ============================================================

import os
import numpy as np
import pandas as pd

STEP4_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "step4_contrast_computation")
TFIDF_STEP4_DIR = os.path.join(STEP4_OUTPUT_DIR, "tfidf")
SCIBERT_STEP4_DIR = os.path.join(STEP4_OUTPUT_DIR, "scibert")

os.makedirs(STEP4_OUTPUT_DIR, exist_ok=True)
os.makedirs(TFIDF_STEP4_DIR, exist_ok=True)
os.makedirs(SCIBERT_STEP4_DIR, exist_ok=True)

# ============================================================
# 4.1 Safety checks
# ============================================================

print("========== STEP 4 SAFETY CHECK ==========")

print("CS docs shape:", cs_docs.shape)
print("TF-IDF CS matrix shape:", X_cs_tfidf.shape)
print("SciBERT CS matrix shape:", E_cs_scibert.shape)

print("\nCS label counts:")
display(cs_docs["label"].value_counts().reset_index())

assert X_cs_tfidf.shape[0] == len(cs_docs), "TF-IDF row count does not match CS documents."
assert E_cs_scibert.shape[0] == len(cs_docs), "SciBERT row count does not match CS documents."
assert int((cs_docs["domain"] == "Medical").sum()) == 0, "Medical documents are present in Step 4 CS data."

cs_ai_mask = cs_docs["label"].values == "AI"
cs_human_mask = cs_docs["label"].values == "Human"

print("\nAI rows:", int(cs_ai_mask.sum()))
print("Human rows:", int(cs_human_mask.sum()))

# ============================================================
# 4.2 TF-IDF contrast computation
# ============================================================

X_cs_tfidf_ai = X_cs_tfidf[cs_ai_mask]
X_cs_tfidf_human = X_cs_tfidf[cs_human_mask]

mean_tfidf_ai = np.asarray(X_cs_tfidf_ai.mean(axis=0)).ravel()
mean_tfidf_human = np.asarray(X_cs_tfidf_human.mean(axis=0)).ravel()

tfidf_contrast_ai_minus_human = mean_tfidf_ai - mean_tfidf_human
tfidf_abs_contrast = np.abs(tfidf_contrast_ai_minus_human)

tfidf_contrast_df = pd.DataFrame({
    "feature_id": np.arange(len(tfidf_feature_names)),
    "feature": tfidf_feature_names,
    "mean_tfidf_ai": mean_tfidf_ai,
    "mean_tfidf_human": mean_tfidf_human,
    "contrast_ai_minus_human": tfidf_contrast_ai_minus_human,
    "abs_contrast": tfidf_abs_contrast,
    "idf": tfidf_vectorizer_cs.idf_
})

tfidf_contrast_df["direction"] = np.where(
    tfidf_contrast_df["contrast_ai_minus_human"] > 0,
    "AI-dominant",
    np.where(
        tfidf_contrast_df["contrast_ai_minus_human"] < 0,
        "Human-dominant",
        "Tie"
    )
)

print("\n========== STEP 4A: TF-IDF CONTRAST RESULT ==========")
print("Total TF-IDF features:", len(tfidf_contrast_df))
print("AI-dominant features:", int((tfidf_contrast_df["contrast_ai_minus_human"] > 0).sum()))
print("Human-dominant features:", int((tfidf_contrast_df["contrast_ai_minus_human"] < 0).sum()))
print("Tie features:", int((tfidf_contrast_df["contrast_ai_minus_human"] == 0).sum()))
print("Minimum contrast:", tfidf_contrast_df["contrast_ai_minus_human"].min())
print("Maximum contrast:", tfidf_contrast_df["contrast_ai_minus_human"].max())
print("Std. deviation:", tfidf_contrast_df["contrast_ai_minus_human"].std())

print("\nTop 10 AI-dominant TF-IDF features:")
display(
    tfidf_contrast_df
    .sort_values("contrast_ai_minus_human", ascending=False)
    .head(10)
)

print("\nTop 10 Human-dominant TF-IDF features:")
display(
    tfidf_contrast_df
    .sort_values("contrast_ai_minus_human", ascending=True)
    .head(10)
)

tfidf_contrast_output_path = os.path.join(
    TFIDF_STEP4_DIR,
    "step4_tfidf_ai_minus_human_contrast.csv"
)

tfidf_contrast_df.to_csv(
    tfidf_contrast_output_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved TF-IDF contrast file:")
print(tfidf_contrast_output_path)

# ============================================================
# 4.3 SciBERT contrast computation
# ============================================================

E_cs_scibert_ai = E_cs_scibert[cs_ai_mask]
E_cs_scibert_human = E_cs_scibert[cs_human_mask]

mean_scibert_ai = E_cs_scibert_ai.mean(axis=0)
mean_scibert_human = E_cs_scibert_human.mean(axis=0)

scibert_contrast_ai_minus_human = mean_scibert_ai - mean_scibert_human
scibert_abs_contrast = np.abs(scibert_contrast_ai_minus_human)

scibert_contrast_df = pd.DataFrame({
    "dimension_id": np.arange(E_cs_scibert.shape[1]),
    "mean_scibert_ai": mean_scibert_ai,
    "mean_scibert_human": mean_scibert_human,
    "contrast_ai_minus_human": scibert_contrast_ai_minus_human,
    "abs_contrast": scibert_abs_contrast
})

scibert_contrast_df["direction"] = np.where(
    scibert_contrast_df["contrast_ai_minus_human"] > 0,
    "AI-dominant",
    np.where(
        scibert_contrast_df["contrast_ai_minus_human"] < 0,
        "Human-dominant",
        "Tie"
    )
)

print("\n========== STEP 4B: SCIBERT CONTRAST RESULT ==========")
print("Total SciBERT dimensions:", len(scibert_contrast_df))
print("AI-dominant dimensions:", int((scibert_contrast_df["contrast_ai_minus_human"] > 0).sum()))
print("Human-dominant dimensions:", int((scibert_contrast_df["contrast_ai_minus_human"] < 0).sum()))
print("Tie dimensions:", int((scibert_contrast_df["contrast_ai_minus_human"] == 0).sum()))
print("Minimum contrast:", scibert_contrast_df["contrast_ai_minus_human"].min())
print("Maximum contrast:", scibert_contrast_df["contrast_ai_minus_human"].max())
print("Std. deviation:", scibert_contrast_df["contrast_ai_minus_human"].std())

print("\nTop 10 AI-dominant SciBERT dimensions:")
display(
    scibert_contrast_df
    .sort_values("contrast_ai_minus_human", ascending=False)
    .head(10)
)

print("\nTop 10 Human-dominant SciBERT dimensions:")
display(
    scibert_contrast_df
    .sort_values("contrast_ai_minus_human", ascending=True)
    .head(10)
)

scibert_contrast_output_path = os.path.join(
    SCIBERT_STEP4_DIR,
    "step4_scibert_ai_minus_human_dimension_contrast.csv"
)

scibert_contrast_df.to_csv(
    scibert_contrast_output_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved SciBERT contrast file:")
print(scibert_contrast_output_path)

# ============================================================
# 4.4 Final Step 4 summary
# ============================================================

step4_summary = pd.DataFrame([
    {
        "representation": "TF-IDF",
        "total_features_or_dimensions": len(tfidf_contrast_df),
        "ai_dominant": int((tfidf_contrast_df["contrast_ai_minus_human"] > 0).sum()),
        "human_dominant": int((tfidf_contrast_df["contrast_ai_minus_human"] < 0).sum()),
        "tie": int((tfidf_contrast_df["contrast_ai_minus_human"] == 0).sum()),
        "min_contrast": tfidf_contrast_df["contrast_ai_minus_human"].min(),
        "max_contrast": tfidf_contrast_df["contrast_ai_minus_human"].max(),
        "std_contrast": tfidf_contrast_df["contrast_ai_minus_human"].std()
    },
    {
        "representation": "SciBERT",
        "total_features_or_dimensions": len(scibert_contrast_df),
        "ai_dominant": int((scibert_contrast_df["contrast_ai_minus_human"] > 0).sum()),
        "human_dominant": int((scibert_contrast_df["contrast_ai_minus_human"] < 0).sum()),
        "tie": int((scibert_contrast_df["contrast_ai_minus_human"] == 0).sum()),
        "min_contrast": scibert_contrast_df["contrast_ai_minus_human"].min(),
        "max_contrast": scibert_contrast_df["contrast_ai_minus_human"].max(),
        "std_contrast": scibert_contrast_df["contrast_ai_minus_human"].std()
    }
])

print("\n========== FINAL STEP 4 SUMMARY ==========")
display(step4_summary)

step4_summary_path = os.path.join(
    STEP4_OUTPUT_DIR,
    "step4_contrast_summary.csv"
)

step4_summary.to_csv(
    step4_summary_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved Step 4 summary:")
print(step4_summary_path)

In [ ]:
# ============================================================
# STEP 5: Contrast Feature Selection
# TF-IDF: Top AI words + Top Human words
# SciBERT: Top AI dimensions + Top Human dimensions
# ============================================================

import os
import re
import numpy as np
import pandas as pd

STEP5_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "step5_contrast_feature_selection")
TFIDF_STEP5_DIR = os.path.join(STEP5_OUTPUT_DIR, "tfidf")
SCIBERT_STEP5_DIR = os.path.join(STEP5_OUTPUT_DIR, "scibert")

os.makedirs(STEP5_OUTPUT_DIR, exist_ok=True)
os.makedirs(TFIDF_STEP5_DIR, exist_ok=True)
os.makedirs(SCIBERT_STEP5_DIR, exist_ok=True)

# ============================================================
# 5.1 Artifact-only filtering for TF-IDF
# This follows the original pipeline:
# remove only obvious technical/non-linguistic artifacts
# ============================================================

def is_artifact_feature(term):
    term = str(term).lower().strip()

    artifact_patterns = [
        r"http",
        r"www",
        r"github",
        r"\.com",
        r"\.org",
        r"\.net",
        r"\burl\b",
        r"\bdoi\b",
        r"\barxiv\b",
        r"\bhtml\b",
        r"\bxml\b",
        r"\bpdf\b",
        r"\blatex\b",
        r"\btex\b",
        r"\bfig\b",
        r"\bfigure\b",
        r"\btable\b",
        r"\bsupplementary\b",
        r"\\",
        r"\$",
        r"[_{}<>]",
    ]

    if len(term) < 2:
        return True

    # Remove terms containing digits because many are version/model/code fragments
    if re.search(r"\d", term):
        return True

    return any(re.search(p, term) for p in artifact_patterns)

tfidf_contrast_filtered_df = tfidf_contrast_df[
    ~tfidf_contrast_df["feature"].apply(is_artifact_feature)
].copy().reset_index(drop=True)

artifact_removed_count = len(tfidf_contrast_df) - len(tfidf_contrast_filtered_df)

print("========== STEP 5A: TF-IDF ARTIFACT FILTERING ==========")
print("Original TF-IDF features:", len(tfidf_contrast_df))
print("Artifact features removed:", artifact_removed_count)
print("Remaining TF-IDF contrast features:", len(tfidf_contrast_filtered_df))

removed_artifacts_df = tfidf_contrast_df[
    tfidf_contrast_df["feature"].apply(is_artifact_feature)
].copy()

print("\nSample removed artifact features:")
display(removed_artifacts_df[["feature_id", "feature", "contrast_ai_minus_human", "direction"]].head(20))

# ============================================================
# 5.2 Select fixed TF-IDF contrast vocabulary
# Top 100 AI-dominant + Top 100 Human-dominant
# ============================================================

TOP_N_AI_TFIDF = 100
TOP_N_HUMAN_TFIDF = 100

top_ai_tfidf_vocab = (
    tfidf_contrast_filtered_df[
        tfidf_contrast_filtered_df["contrast_ai_minus_human"] > 0
    ]
    .sort_values("contrast_ai_minus_human", ascending=False)
    .head(TOP_N_AI_TFIDF)
    .copy()
)

top_human_tfidf_vocab = (
    tfidf_contrast_filtered_df[
        tfidf_contrast_filtered_df["contrast_ai_minus_human"] < 0
    ]
    .sort_values("contrast_ai_minus_human", ascending=True)
    .head(TOP_N_HUMAN_TFIDF)
    .copy()
)

top_ai_tfidf_vocab["vocab_group"] = "AI"
top_human_tfidf_vocab["vocab_group"] = "Human"

selected_tfidf_vocab = pd.concat(
    [top_ai_tfidf_vocab, top_human_tfidf_vocab],
    ignore_index=True
)

selected_tfidf_feature_ids = selected_tfidf_vocab["feature_id"].values
selected_tfidf_terms = selected_tfidf_vocab["feature"].values

tfidf_ai_terms = set(top_ai_tfidf_vocab["feature"])
tfidf_human_terms = set(top_human_tfidf_vocab["feature"])

print("\n========== STEP 5B: TF-IDF CONTRAST VOCABULARY SELECTION ==========")
print("Selected AI contrast vocabulary:", len(top_ai_tfidf_vocab))
print("Selected Human contrast vocabulary:", len(top_human_tfidf_vocab))
print("Total contrast vocabulary size:", len(selected_tfidf_vocab))
print("Duplicate selected features:", int(selected_tfidf_vocab["feature"].duplicated().sum()))
print("AI/Human overlap:", len(tfidf_ai_terms.intersection(tfidf_human_terms)))
print("All AI terms positive:", bool((top_ai_tfidf_vocab["contrast_ai_minus_human"] > 0).all()))
print("All Human terms negative:", bool((top_human_tfidf_vocab["contrast_ai_minus_human"] < 0).all()))

print("\nTop 20 selected AI TF-IDF terms:")
display(
    top_ai_tfidf_vocab[
        ["feature_id", "feature", "contrast_ai_minus_human", "mean_tfidf_ai", "mean_tfidf_human", "idf"]
    ].head(20)
)

print("\nTop 20 selected Human TF-IDF terms:")
display(
    top_human_tfidf_vocab[
        ["feature_id", "feature", "contrast_ai_minus_human", "mean_tfidf_ai", "mean_tfidf_human", "idf"]
    ].head(20)
)

# Save TF-IDF Step 5 outputs
tfidf_filtered_path = os.path.join(TFIDF_STEP5_DIR, "step5_tfidf_contrast_after_artifact_filtering.csv")
tfidf_selected_vocab_path = os.path.join(TFIDF_STEP5_DIR, "step5_selected_tfidf_200_contrast_vocabulary.csv")
tfidf_top_ai_path = os.path.join(TFIDF_STEP5_DIR, "step5_top_100_ai_tfidf_terms.csv")
tfidf_top_human_path = os.path.join(TFIDF_STEP5_DIR, "step5_top_100_human_tfidf_terms.csv")

tfidf_contrast_filtered_df.to_csv(tfidf_filtered_path, index=False, encoding="utf-8-sig")
selected_tfidf_vocab.to_csv(tfidf_selected_vocab_path, index=False, encoding="utf-8-sig")
top_ai_tfidf_vocab.to_csv(tfidf_top_ai_path, index=False, encoding="utf-8-sig")
top_human_tfidf_vocab.to_csv(tfidf_top_human_path, index=False, encoding="utf-8-sig")

print("\nSaved TF-IDF Step 5 outputs:")
print(tfidf_filtered_path)
print(tfidf_selected_vocab_path)
print(tfidf_top_ai_path)
print(tfidf_top_human_path)

# ============================================================
# 5.3 Select fixed SciBERT contrast dimensions
# SciBERT has no word vocabulary, so we select dimensions instead.
# Same contrast idea: AI-dominant dimensions + Human-dominant dimensions
# ============================================================

TOP_N_AI_SCIBERT = 100
TOP_N_HUMAN_SCIBERT = 100

top_ai_scibert_dims = (
    scibert_contrast_df[
        scibert_contrast_df["contrast_ai_minus_human"] > 0
    ]
    .sort_values("contrast_ai_minus_human", ascending=False)
    .head(TOP_N_AI_SCIBERT)
    .copy()
)

top_human_scibert_dims = (
    scibert_contrast_df[
        scibert_contrast_df["contrast_ai_minus_human"] < 0
    ]
    .sort_values("contrast_ai_minus_human", ascending=True)
    .head(TOP_N_HUMAN_SCIBERT)
    .copy()
)

top_ai_scibert_dims["feature_group"] = "AI"
top_human_scibert_dims["feature_group"] = "Human"

selected_scibert_features = pd.concat(
    [top_ai_scibert_dims, top_human_scibert_dims],
    ignore_index=True
)

selected_scibert_dim_ids = selected_scibert_features["dimension_id"].values

scibert_ai_dims = set(top_ai_scibert_dims["dimension_id"])
scibert_human_dims = set(top_human_scibert_dims["dimension_id"])

print("\n========== STEP 5C: SCIBERT CONTRAST DIMENSION SELECTION ==========")
print("Selected AI-dominant SciBERT dimensions:", len(top_ai_scibert_dims))
print("Selected Human-dominant SciBERT dimensions:", len(top_human_scibert_dims))
print("Total selected SciBERT dimensions:", len(selected_scibert_features))
print("Duplicate selected dimensions:", int(selected_scibert_features["dimension_id"].duplicated().sum()))
print("AI/Human dimension overlap:", len(scibert_ai_dims.intersection(scibert_human_dims)))
print("All AI dimensions positive:", bool((top_ai_scibert_dims["contrast_ai_minus_human"] > 0).all()))
print("All Human dimensions negative:", bool((top_human_scibert_dims["contrast_ai_minus_human"] < 0).all()))

print("\nTop 20 selected AI SciBERT dimensions:")
display(
    top_ai_scibert_dims[
        ["dimension_id", "contrast_ai_minus_human", "mean_scibert_ai", "mean_scibert_human"]
    ].head(20)
)

print("\nTop 20 selected Human SciBERT dimensions:")
display(
    top_human_scibert_dims[
        ["dimension_id", "contrast_ai_minus_human", "mean_scibert_ai", "mean_scibert_human"]
    ].head(20)
)

# Save SciBERT Step 5 outputs
scibert_selected_path = os.path.join(SCIBERT_STEP5_DIR, "step5_selected_scibert_200_contrast_dimensions.csv")
scibert_top_ai_path = os.path.join(SCIBERT_STEP5_DIR, "step5_top_100_ai_scibert_dimensions.csv")
scibert_top_human_path = os.path.join(SCIBERT_STEP5_DIR, "step5_top_100_human_scibert_dimensions.csv")

selected_scibert_features.to_csv(scibert_selected_path, index=False, encoding="utf-8-sig")
top_ai_scibert_dims.to_csv(scibert_top_ai_path, index=False, encoding="utf-8-sig")
top_human_scibert_dims.to_csv(scibert_top_human_path, index=False, encoding="utf-8-sig")

print("\nSaved SciBERT Step 5 outputs:")
print(scibert_selected_path)
print(scibert_top_ai_path)
print(scibert_top_human_path)

# ============================================================
# 5.4 Final Step 5 summary
# ============================================================

step5_summary = pd.DataFrame([
    {
        "representation": "TF-IDF",
        "original_features_or_dimensions": len(tfidf_contrast_df),
        "artifact_removed": artifact_removed_count,
        "remaining_after_filtering": len(tfidf_contrast_filtered_df),
        "selected_ai": len(top_ai_tfidf_vocab),
        "selected_human": len(top_human_tfidf_vocab),
        "total_selected": len(selected_tfidf_vocab),
        "duplicate_selected": int(selected_tfidf_vocab["feature"].duplicated().sum()),
        "ai_human_overlap": len(tfidf_ai_terms.intersection(tfidf_human_terms)),
        "all_ai_positive": bool((top_ai_tfidf_vocab["contrast_ai_minus_human"] > 0).all()),
        "all_human_negative": bool((top_human_tfidf_vocab["contrast_ai_minus_human"] < 0).all())
    },
    {
        "representation": "SciBERT",
        "original_features_or_dimensions": len(scibert_contrast_df),
        "artifact_removed": "Not applicable",
        "remaining_after_filtering": len(scibert_contrast_df),
        "selected_ai": len(top_ai_scibert_dims),
        "selected_human": len(top_human_scibert_dims),
        "total_selected": len(selected_scibert_features),
        "duplicate_selected": int(selected_scibert_features["dimension_id"].duplicated().sum()),
        "ai_human_overlap": len(scibert_ai_dims.intersection(scibert_human_dims)),
        "all_ai_positive": bool((top_ai_scibert_dims["contrast_ai_minus_human"] > 0).all()),
        "all_human_negative": bool((top_human_scibert_dims["contrast_ai_minus_human"] < 0).all())
    }
])

print("\n========== FINAL STEP 5 SUMMARY ==========")
display(step5_summary)

step5_summary_path = os.path.join(STEP5_OUTPUT_DIR, "step5_feature_selection_summary.csv")
step5_summary.to_csv(step5_summary_path, index=False, encoding="utf-8-sig")

print("\nSaved Step 5 summary:")
print(step5_summary_path)

In [ ]:
# ============================================================
# STEP 5.1: Correct TF-IDF artifact filtering
# Run this after Step 5 and before Step 6
# SciBERT selected dimensions stay unchanged
# ============================================================

import re
import os
import pandas as pd
import numpy as np

STEP5_FIX_DIR = os.path.join(STEP5_OUTPUT_DIR, "tfidf_corrected_filter")
os.makedirs(STEP5_FIX_DIR, exist_ok=True)

def is_artifact_feature_strict(term):
    term = str(term).lower().strip()

    artifact_patterns = [
        # URL / web fragments
        r"http",
        r"www",
        r"github",
        r"huggingface",
        r"\bcom\b",
        r"\borg\b",
        r"\bnet\b",
        r"\bio\b",
        r"\burl\b",
        r"\bdoi\b",
        r"\barxiv\b",

        # HTML/XML/PDF/LaTeX fragments
        r"\bhtml\b",
        r"\bxml\b",
        r"\bpdf\b",
        r"\blatex\b",
        r"\btex\b",
        r"\btextbf\b",
        r"\btextit\b",
        r"\bemph\b",
        r"\bmathbf\b",
        r"\bmathbb\b",
        r"\bmathrm\b",
        r"\bhref\b",

        # document/formatting fragments
        r"\bfig\b",
        r"\bfigure\b",
        r"\btable\b",
        r"\bsupplementary\b",
        r"\bappendix\b",

        # symbols often created from markup
        r"\\",
        r"\$",
        r"[_{}<>]",
    ]

    if len(term) < 2:
        return True

    # Remove terms containing digits, usually model/version/code fragments
    if re.search(r"\d", term):
        return True

    # Remove if any token inside a bigram is an artifact
    tokens = term.split()
    artifact_single_tokens = {
        "com", "org", "net", "io", "http", "www", "github",
        "html", "xml", "pdf", "latex", "tex", "textbf",
        "textit", "emph", "mathbf", "mathbb", "mathrm",
        "href", "url", "doi", "arxiv"
    }

    if any(tok in artifact_single_tokens for tok in tokens):
        return True

    return any(re.search(p, term) for p in artifact_patterns)


# Apply corrected filtering
tfidf_contrast_filtered_df = tfidf_contrast_df[
    ~tfidf_contrast_df["feature"].apply(is_artifact_feature_strict)
].copy().reset_index(drop=True)

removed_artifacts_corrected_df = tfidf_contrast_df[
    tfidf_contrast_df["feature"].apply(is_artifact_feature_strict)
].copy()

artifact_removed_count = len(tfidf_contrast_df) - len(tfidf_contrast_filtered_df)

print("========== STEP 5.1A: CORRECTED TF-IDF ARTIFACT FILTERING ==========")
print("Original TF-IDF features:", len(tfidf_contrast_df))
print("Artifact features removed:", artifact_removed_count)
print("Remaining TF-IDF contrast features:", len(tfidf_contrast_filtered_df))

print("\nSample removed artifact features:")
display(
    removed_artifacts_corrected_df[
        ["feature_id", "feature", "contrast_ai_minus_human", "direction"]
    ].head(40)
)

# Reselect top 100 AI and top 100 Human terms
TOP_N_AI_TFIDF = 100
TOP_N_HUMAN_TFIDF = 100

top_ai_tfidf_vocab = (
    tfidf_contrast_filtered_df[
        tfidf_contrast_filtered_df["contrast_ai_minus_human"] > 0
    ]
    .sort_values("contrast_ai_minus_human", ascending=False)
    .head(TOP_N_AI_TFIDF)
    .copy()
)

top_human_tfidf_vocab = (
    tfidf_contrast_filtered_df[
        tfidf_contrast_filtered_df["contrast_ai_minus_human"] < 0
    ]
    .sort_values("contrast_ai_minus_human", ascending=True)
    .head(TOP_N_HUMAN_TFIDF)
    .copy()
)

top_ai_tfidf_vocab["vocab_group"] = "AI"
top_human_tfidf_vocab["vocab_group"] = "Human"

selected_tfidf_vocab = pd.concat(
    [top_ai_tfidf_vocab, top_human_tfidf_vocab],
    ignore_index=True
)

selected_tfidf_feature_ids = selected_tfidf_vocab["feature_id"].values
selected_tfidf_terms = selected_tfidf_vocab["feature"].values

tfidf_ai_terms = set(top_ai_tfidf_vocab["feature"])
tfidf_human_terms = set(top_human_tfidf_vocab["feature"])

print("\n========== STEP 5.1B: CORRECTED TF-IDF VOCABULARY SELECTION ==========")
print("Selected AI contrast vocabulary:", len(top_ai_tfidf_vocab))
print("Selected Human contrast vocabulary:", len(top_human_tfidf_vocab))
print("Total contrast vocabulary size:", len(selected_tfidf_vocab))
print("Duplicate selected features:", int(selected_tfidf_vocab["feature"].duplicated().sum()))
print("AI/Human overlap:", len(tfidf_ai_terms.intersection(tfidf_human_terms)))
print("All AI terms positive:", bool((top_ai_tfidf_vocab["contrast_ai_minus_human"] > 0).all()))
print("All Human terms negative:", bool((top_human_tfidf_vocab["contrast_ai_minus_human"] < 0).all()))

print("\nTop 20 corrected AI TF-IDF terms:")
display(
    top_ai_tfidf_vocab[
        ["feature_id", "feature", "contrast_ai_minus_human", "mean_tfidf_ai", "mean_tfidf_human", "idf"]
    ].head(20)
)

print("\nTop 20 corrected Human TF-IDF terms:")
display(
    top_human_tfidf_vocab[
        ["feature_id", "feature", "contrast_ai_minus_human", "mean_tfidf_ai", "mean_tfidf_human", "idf"]
    ].head(20)
)

# Check whether unwanted artifacts remain in selected vocabulary
artifact_check_terms = ["http", "github", "com", "textbf", "html", "pdf", "latex", "href"]

print("\n========== SELECTED VOCABULARY ARTIFACT CHECK ==========")
for bad in artifact_check_terms:
    count = selected_tfidf_vocab["feature"].str.lower().str.contains(bad, regex=False, na=False).sum()
    print(f"{bad}: {count}")

# Save corrected outputs
corrected_filtered_path = os.path.join(
    STEP5_FIX_DIR,
    "step5_1_tfidf_contrast_after_corrected_artifact_filtering.csv"
)

corrected_vocab_path = os.path.join(
    STEP5_FIX_DIR,
    "step5_1_selected_tfidf_200_corrected_contrast_vocabulary.csv"
)

corrected_top_ai_path = os.path.join(
    STEP5_FIX_DIR,
    "step5_1_top_100_ai_tfidf_terms_corrected.csv"
)

corrected_top_human_path = os.path.join(
    STEP5_FIX_DIR,
    "step5_1_top_100_human_tfidf_terms_corrected.csv"
)

tfidf_contrast_filtered_df.to_csv(corrected_filtered_path, index=False, encoding="utf-8-sig")
selected_tfidf_vocab.to_csv(corrected_vocab_path, index=False, encoding="utf-8-sig")
top_ai_tfidf_vocab.to_csv(corrected_top_ai_path, index=False, encoding="utf-8-sig")
top_human_tfidf_vocab.to_csv(corrected_top_human_path, index=False, encoding="utf-8-sig")

print("\nSaved corrected TF-IDF Step 5.1 outputs:")
print(corrected_filtered_path)
print(corrected_vocab_path)
print(corrected_top_ai_path)
print(corrected_top_human_path)

In [ ]:
# ============================================================
# STEP 6: Domain A / CS Feature Matrix Construction
# TF-IDF DTM using selected 200 contrast vocabulary
# SciBERT matrix using selected 200 contrast dimensions
# ============================================================

import os
import numpy as np
import pandas as pd

from scipy import sparse
from sklearn.preprocessing import normalize

STEP6_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "step6_domain_a_feature_matrix")
TFIDF_STEP6_DIR = os.path.join(STEP6_OUTPUT_DIR, "tfidf")
SCIBERT_STEP6_DIR = os.path.join(STEP6_OUTPUT_DIR, "scibert")

os.makedirs(STEP6_OUTPUT_DIR, exist_ok=True)
os.makedirs(TFIDF_STEP6_DIR, exist_ok=True)
os.makedirs(SCIBERT_STEP6_DIR, exist_ok=True)

# ============================================================
# 6.1 Safety checks
# ============================================================

print("========== STEP 6 SAFETY CHECK ==========")

print("CS documents:", len(cs_docs))
print("TF-IDF full CS matrix shape:", X_cs_tfidf.shape)
print("SciBERT full CS matrix shape:", E_cs_scibert.shape)

print("\nSelected TF-IDF feature IDs:", len(selected_tfidf_feature_ids))
print("Selected TF-IDF terms:", len(selected_tfidf_terms))
print("Selected SciBERT dimension IDs:", len(selected_scibert_dim_ids))

assert len(cs_docs) == X_cs_tfidf.shape[0], "CS docs and TF-IDF rows do not match."
assert len(cs_docs) == E_cs_scibert.shape[0], "CS docs and SciBERT rows do not match."
assert len(selected_tfidf_feature_ids) == 200, "TF-IDF selected vocabulary should be 200."
assert len(selected_scibert_dim_ids) == 200, "SciBERT selected dimensions should be 200."
assert int((cs_docs["domain"] == "Medical").sum()) == 0, "Medical documents found in Step 6 CS data."

cs_ai_mask = cs_docs["label"].values == "AI"
cs_human_mask = cs_docs["label"].values == "Human"

print("\nCS AI documents:", int(cs_ai_mask.sum()))
print("CS Human documents:", int(cs_human_mask.sum()))

# ============================================================
# 6.2 TF-IDF DTM construction for Domain A / CS
# Select only the corrected 200 contrast-vocabulary columns
# ============================================================

X_cs_tfidf_dtm = X_cs_tfidf[:, selected_tfidf_feature_ids]

# Normalize selected DTM before UMAP/clustering
X_cs_tfidf_dtm_norm = normalize(X_cs_tfidf_dtm, norm="l2", axis=1)

tfidf_nonzero_per_doc = np.diff(X_cs_tfidf_dtm.indptr)
tfidf_nonzero_ai = tfidf_nonzero_per_doc[cs_ai_mask]
tfidf_nonzero_human = tfidf_nonzero_per_doc[cs_human_mask]

print("\n========== STEP 6A: TF-IDF DTM DOMAIN A / CS ==========")
print("DTM matrix shape:", X_cs_tfidf_dtm.shape)
print("Rows/documents:", X_cs_tfidf_dtm.shape[0])
print("Vocabulary terms:", X_cs_tfidf_dtm.shape[1])
print("Non-zero DTM values:", X_cs_tfidf_dtm.nnz)
print("Sparsity:", round(1 - (X_cs_tfidf_dtm.nnz / (X_cs_tfidf_dtm.shape[0] * X_cs_tfidf_dtm.shape[1])), 6))
print("All-zero rows:", int((tfidf_nonzero_per_doc == 0).sum()))
print("AI avg non-zero terms/doc:", round(float(tfidf_nonzero_ai.mean()), 4))
print("Human avg non-zero terms/doc:", round(float(tfidf_nonzero_human.mean()), 4))
print("AI all-zero rows:", int((tfidf_nonzero_ai == 0).sum()))
print("Human all-zero rows:", int((tfidf_nonzero_human == 0).sum()))

print("\nSelected TF-IDF vocabulary preview:")
display(
    selected_tfidf_vocab[
        ["vocab_group", "feature_id", "feature", "contrast_ai_minus_human"]
    ].head(10)
)

# Save TF-IDF Step 6 outputs
tfidf_dtm_path = os.path.join(TFIDF_STEP6_DIR, "step6_X_cs_tfidf_200_dtm.npz")
tfidf_dtm_norm_path = os.path.join(TFIDF_STEP6_DIR, "step6_X_cs_tfidf_200_dtm_l2_normalized.npz")
tfidf_vocab_path = os.path.join(TFIDF_STEP6_DIR, "step6_selected_tfidf_vocabulary_used.csv")

sparse.save_npz(tfidf_dtm_path, X_cs_tfidf_dtm)
sparse.save_npz(tfidf_dtm_norm_path, X_cs_tfidf_dtm_norm)
selected_tfidf_vocab.to_csv(tfidf_vocab_path, index=False, encoding="utf-8-sig")

print("\nSaved TF-IDF Step 6 outputs:")
print(tfidf_dtm_path)
print(tfidf_dtm_norm_path)
print(tfidf_vocab_path)

# ============================================================
# 6.3 SciBERT selected feature matrix for Domain A / CS
# Select only the 200 CS-derived contrast dimensions
# ============================================================

X_cs_scibert_selected = E_cs_scibert[:, selected_scibert_dim_ids]

# Normalize selected SciBERT dimensions before UMAP/clustering
X_cs_scibert_selected_norm = normalize(X_cs_scibert_selected, norm="l2", axis=1)

print("\n========== STEP 6B: SCIBERT FEATURE MATRIX DOMAIN A / CS ==========")
print("Selected SciBERT matrix shape:", X_cs_scibert_selected.shape)
print("Rows/documents:", X_cs_scibert_selected.shape[0])
print("Selected dimensions:", X_cs_scibert_selected.shape[1])
print("NaN values:", int(np.isnan(X_cs_scibert_selected).sum()))
print("Infinite values:", int(np.isinf(X_cs_scibert_selected).sum()))
print("Minimum value:", float(np.min(X_cs_scibert_selected)))
print("Maximum value:", float(np.max(X_cs_scibert_selected)))
print("Mean value:", float(np.mean(X_cs_scibert_selected)))
print("Std value:", float(np.std(X_cs_scibert_selected)))

print("\nSelected SciBERT dimensions preview:")
display(
    selected_scibert_features[
        ["feature_group", "dimension_id", "contrast_ai_minus_human"]
    ].head(10)
)

# Save SciBERT Step 6 outputs
scibert_selected_path = os.path.join(SCIBERT_STEP6_DIR, "step6_X_cs_scibert_200_selected_dimensions.npy")
scibert_selected_norm_path = os.path.join(SCIBERT_STEP6_DIR, "step6_X_cs_scibert_200_selected_dimensions_l2_normalized.npy")
scibert_dims_path = os.path.join(SCIBERT_STEP6_DIR, "step6_selected_scibert_dimensions_used.csv")

np.save(scibert_selected_path, X_cs_scibert_selected)
np.save(scibert_selected_norm_path, X_cs_scibert_selected_norm)
selected_scibert_features.to_csv(scibert_dims_path, index=False, encoding="utf-8-sig")

print("\nSaved SciBERT Step 6 outputs:")
print(scibert_selected_path)
print(scibert_selected_norm_path)
print(scibert_dims_path)

# ============================================================
# 6.4 Final Step 6 summary
# ============================================================

step6_summary = pd.DataFrame([
    {
        "representation": "TF-IDF",
        "domain": "Domain A / CS",
        "matrix_shape": str(X_cs_tfidf_dtm.shape),
        "documents": X_cs_tfidf_dtm.shape[0],
        "features_or_dimensions": X_cs_tfidf_dtm.shape[1],
        "nonzero_values": X_cs_tfidf_dtm.nnz,
        "sparsity": round(1 - (X_cs_tfidf_dtm.nnz / (X_cs_tfidf_dtm.shape[0] * X_cs_tfidf_dtm.shape[1])), 6),
        "all_zero_rows": int((tfidf_nonzero_per_doc == 0).sum()),
        "ai_avg_nonzero": round(float(tfidf_nonzero_ai.mean()), 4),
        "human_avg_nonzero": round(float(tfidf_nonzero_human.mean()), 4)
    },
    {
        "representation": "SciBERT",
        "domain": "Domain A / CS",
        "matrix_shape": str(X_cs_scibert_selected.shape),
        "documents": X_cs_scibert_selected.shape[0],
        "features_or_dimensions": X_cs_scibert_selected.shape[1],
        "nonzero_values": "Dense matrix",
        "sparsity": "Not applicable",
        "all_zero_rows": int((np.linalg.norm(X_cs_scibert_selected, axis=1) == 0).sum()),
        "ai_avg_nonzero": "Not applicable",
        "human_avg_nonzero": "Not applicable"
    }
])

print("\n========== FINAL STEP 6 SUMMARY ==========")
display(step6_summary)

step6_summary_path = os.path.join(STEP6_OUTPUT_DIR, "step6_domain_a_feature_matrix_summary.csv")
step6_summary.to_csv(step6_summary_path, index=False, encoding="utf-8-sig")

print("\nSaved Step 6 summary:")
print(step6_summary_path)

In [ ]:
# ============================================================
# STEP 7: UMAP + Clustering on Domain A / CS
# TF-IDF + UMAP + clustering
# SciBERT + UMAP + clustering
# ============================================================

!pip install -q umap-learn

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

import umap.umap_ as umap

from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score
from sklearn.metrics import confusion_matrix

STEP7_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "step7_domain_a_umap_clustering")
TFIDF_STEP7_DIR = os.path.join(STEP7_OUTPUT_DIR, "tfidf")
SCIBERT_STEP7_DIR = os.path.join(STEP7_OUTPUT_DIR, "scibert")

os.makedirs(STEP7_OUTPUT_DIR, exist_ok=True)
os.makedirs(TFIDF_STEP7_DIR, exist_ok=True)
os.makedirs(SCIBERT_STEP7_DIR, exist_ok=True)

# ============================================================
# 7.1 Evaluation helper functions
# ============================================================

def label_to_int(labels):
    """
    Human = 0
    AI = 1
    """
    return np.array([1 if x == "AI" else 0 for x in labels])

def purity_score(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    total_correct = 0

    for cluster_id in np.unique(y_pred):
        idx = y_pred == cluster_id
        cluster_true_labels = y_true[idx]

        if len(cluster_true_labels) == 0:
            continue

        values, counts = np.unique(cluster_true_labels, return_counts=True)
        total_correct += counts.max()

    return total_correct / len(y_true)

def cluster_composition_table(true_labels, pred_labels):
    table = pd.crosstab(
        pd.Series(pred_labels, name="cluster"),
        pd.Series(true_labels, name="true_label")
    )

    # Add majority label and purity per cluster
    rows = []

    for cluster_id in sorted(np.unique(pred_labels)):
        idx = pred_labels == cluster_id
        cluster_labels = np.asarray(true_labels)[idx]

        values, counts = np.unique(cluster_labels, return_counts=True)
        majority_label = values[np.argmax(counts)]
        cluster_purity = counts.max() / counts.sum()

        rows.append({
            "cluster": cluster_id,
            "cluster_size": int(idx.sum()),
            "majority_label": majority_label,
            "cluster_purity": cluster_purity
        })

    summary = pd.DataFrame(rows)

    return table, summary

def evaluate_clustering(X, true_labels, pred_labels, representation_name, domain_name, algorithm_name):
    y_true_int = label_to_int(true_labels)

    purity = purity_score(y_true_int, pred_labels)
    ari = adjusted_rand_score(y_true_int, pred_labels)
    nmi = normalized_mutual_info_score(y_true_int, pred_labels)

    try:
        sil = silhouette_score(X, pred_labels)
    except Exception:
        sil = np.nan

    return {
        "representation": representation_name,
        "domain": domain_name,
        "algorithm": algorithm_name,
        "documents": len(true_labels),
        "human_docs": int((np.asarray(true_labels) == "Human").sum()),
        "ai_docs": int((np.asarray(true_labels) == "AI").sum()),
        "purity": purity,
        "ari": ari,
        "nmi": nmi,
        "silhouette": sil
    }

def run_clustering_suite(X, true_labels, representation_name, domain_name):
    """
    Main algorithm = KMeans
    Additional algorithms = Agglomerative, GMM
    """
    results = []
    predictions = {}

    # 1. KMeans
    kmeans = KMeans(
        n_clusters=2,
        random_state=RANDOM_STATE,
        n_init=30
    )
    pred_kmeans = kmeans.fit_predict(X)
    predictions["KMeans"] = pred_kmeans
    results.append(
        evaluate_clustering(
            X, true_labels, pred_kmeans,
            representation_name, domain_name, "KMeans"
        )
    )

    # 2. Agglomerative clustering
    agg = AgglomerativeClustering(
        n_clusters=2,
        linkage="ward"
    )
    pred_agg = agg.fit_predict(X)
    predictions["Agglomerative"] = pred_agg
    results.append(
        evaluate_clustering(
            X, true_labels, pred_agg,
            representation_name, domain_name, "Agglomerative"
        )
    )

    # 3. Gaussian Mixture
    gmm = GaussianMixture(
        n_components=2,
        random_state=RANDOM_STATE,
        covariance_type="full"
    )
    pred_gmm = gmm.fit_predict(X)
    predictions["GMM"] = pred_gmm
    results.append(
        evaluate_clustering(
            X, true_labels, pred_gmm,
            representation_name, domain_name, "GMM"
        )
    )

    metrics_df = pd.DataFrame(results)

    return metrics_df, predictions

def plot_umap_2d(Z, labels, title, save_path):
    plt.figure(figsize=(7, 5))

    labels = np.asarray(labels)

    for lab in np.unique(labels):
        idx = labels == lab
        plt.scatter(
            Z[idx, 0],
            Z[idx, 1],
            s=8,
            alpha=0.65,
            label=str(lab)
        )

    plt.title(title)
    plt.xlabel("UMAP-1")
    plt.ylabel("UMAP-2")
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.show()

# ============================================================
# 7.2 Safety check
# ============================================================

print("========== STEP 7 SAFETY CHECK ==========")

print("CS documents:", len(cs_docs))
print("TF-IDF selected normalized matrix shape:", X_cs_tfidf_dtm_norm.shape)
print("SciBERT selected normalized matrix shape:", X_cs_scibert_selected_norm.shape)

assert X_cs_tfidf_dtm_norm.shape == (4000, 200), "TF-IDF CS DTM should be (4000, 200)."
assert X_cs_scibert_selected_norm.shape == (4000, 200), "SciBERT CS matrix should be (4000, 200)."

true_labels_cs = cs_docs["label"].values

print("\nTrue label distribution:")
display(pd.Series(true_labels_cs).value_counts().reset_index())

# ============================================================
# 7.3 TF-IDF + UMAP on Domain A / CS
# UMAP is fitted on CS because CS is the source domain.
# This same reducer will later be used to transform Medical.
# ============================================================

print("\n========== STEP 7A: TF-IDF + UMAP DOMAIN A / CS ==========")

tfidf_umap_reducer = umap.UMAP(
    n_components=10,
    n_neighbors=15,
    min_dist=0.05,
    metric="cosine",
    random_state=RANDOM_STATE
)

Z_cs_tfidf_umap = tfidf_umap_reducer.fit_transform(X_cs_tfidf_dtm_norm)

print("Original TF-IDF DTM shape:", X_cs_tfidf_dtm_norm.shape)
print("UMAP-reduced TF-IDF shape:", Z_cs_tfidf_umap.shape)
print("NaN values:", int(np.isnan(Z_cs_tfidf_umap).sum()))
print("Infinite values:", int(np.isinf(Z_cs_tfidf_umap).sum()))

tfidf_cs_metrics, tfidf_cs_predictions = run_clustering_suite(
    Z_cs_tfidf_umap,
    true_labels_cs,
    representation_name="TF-IDF + UMAP",
    domain_name="Domain A / CS"
)

print("\nTF-IDF + UMAP clustering metrics:")
display(tfidf_cs_metrics)

for algo_name, pred in tfidf_cs_predictions.items():
    print(f"\nTF-IDF + UMAP | {algo_name} cluster composition:")
    comp_table, comp_summary = cluster_composition_table(true_labels_cs, pred)
    display(comp_table)
    display(comp_summary)

# Save TF-IDF UMAP reducer and outputs
tfidf_umap_reducer_path = os.path.join(TFIDF_STEP7_DIR, "step7_tfidf_umap_reducer_fitted_on_cs.joblib")
tfidf_umap_matrix_path = os.path.join(TFIDF_STEP7_DIR, "step7_Z_cs_tfidf_umap.npy")
tfidf_metrics_path = os.path.join(TFIDF_STEP7_DIR, "step7_tfidf_cs_clustering_metrics.csv")

joblib.dump(tfidf_umap_reducer, tfidf_umap_reducer_path)
np.save(tfidf_umap_matrix_path, Z_cs_tfidf_umap)
tfidf_cs_metrics.to_csv(tfidf_metrics_path, index=False, encoding="utf-8-sig")

for algo_name, pred in tfidf_cs_predictions.items():
    pred_path = os.path.join(TFIDF_STEP7_DIR, f"step7_tfidf_cs_{algo_name}_clusters.csv")

    pred_df = cs_docs[["doc_id", "domain", "label", "title"]].copy()
    pred_df["cluster"] = pred

    pred_df.to_csv(pred_path, index=False, encoding="utf-8-sig")

print("\nSaved TF-IDF Step 7 outputs:")
print(tfidf_umap_reducer_path)
print(tfidf_umap_matrix_path)
print(tfidf_metrics_path)

# Plot TF-IDF UMAP first two dimensions
plot_umap_2d(
    Z_cs_tfidf_umap,
    true_labels_cs,
    "TF-IDF + UMAP | Domain A / CS by true label",
    os.path.join(TFIDF_STEP7_DIR, "step7_tfidf_umap_cs_by_true_label.png")
)

plot_umap_2d(
    Z_cs_tfidf_umap,
    tfidf_cs_predictions["KMeans"],
    "TF-IDF + UMAP | Domain A / CS by KMeans cluster",
    os.path.join(TFIDF_STEP7_DIR, "step7_tfidf_umap_cs_by_kmeans_cluster.png")
)

# ============================================================
# 7.4 SciBERT + UMAP on Domain A / CS
# UMAP is fitted on CS. Same reducer will later transform Medical.
# ============================================================

print("\n========== STEP 7B: SCIBERT + UMAP DOMAIN A / CS ==========")

scibert_umap_reducer = umap.UMAP(
    n_components=10,
    n_neighbors=15,
    min_dist=0.05,
    metric="cosine",
    random_state=RANDOM_STATE
)

Z_cs_scibert_umap = scibert_umap_reducer.fit_transform(X_cs_scibert_selected_norm)

print("Original SciBERT selected matrix shape:", X_cs_scibert_selected_norm.shape)
print("UMAP-reduced SciBERT shape:", Z_cs_scibert_umap.shape)
print("NaN values:", int(np.isnan(Z_cs_scibert_umap).sum()))
print("Infinite values:", int(np.isinf(Z_cs_scibert_umap).sum()))

scibert_cs_metrics, scibert_cs_predictions = run_clustering_suite(
    Z_cs_scibert_umap,
    true_labels_cs,
    representation_name="SciBERT + UMAP",
    domain_name="Domain A / CS"
)

print("\nSciBERT + UMAP clustering metrics:")
display(scibert_cs_metrics)

for algo_name, pred in scibert_cs_predictions.items():
    print(f"\nSciBERT + UMAP | {algo_name} cluster composition:")
    comp_table, comp_summary = cluster_composition_table(true_labels_cs, pred)
    display(comp_table)
    display(comp_summary)

# Save SciBERT UMAP reducer and outputs
scibert_umap_reducer_path = os.path.join(SCIBERT_STEP7_DIR, "step7_scibert_umap_reducer_fitted_on_cs.joblib")
scibert_umap_matrix_path = os.path.join(SCIBERT_STEP7_DIR, "step7_Z_cs_scibert_umap.npy")
scibert_metrics_path = os.path.join(SCIBERT_STEP7_DIR, "step7_scibert_cs_clustering_metrics.csv")

joblib.dump(scibert_umap_reducer, scibert_umap_reducer_path)
np.save(scibert_umap_matrix_path, Z_cs_scibert_umap)
scibert_cs_metrics.to_csv(scibert_metrics_path, index=False, encoding="utf-8-sig")

for algo_name, pred in scibert_cs_predictions.items():
    pred_path = os.path.join(SCIBERT_STEP7_DIR, f"step7_scibert_cs_{algo_name}_clusters.csv")

    pred_df = cs_docs[["doc_id", "domain", "label", "title"]].copy()
    pred_df["cluster"] = pred

    pred_df.to_csv(pred_path, index=False, encoding="utf-8-sig")

print("\nSaved SciBERT Step 7 outputs:")
print(scibert_umap_reducer_path)
print(scibert_umap_matrix_path)
print(scibert_metrics_path)

# Plot SciBERT UMAP first two dimensions
plot_umap_2d(
    Z_cs_scibert_umap,
    true_labels_cs,
    "SciBERT + UMAP | Domain A / CS by true label",
    os.path.join(SCIBERT_STEP7_DIR, "step7_scibert_umap_cs_by_true_label.png")
)

plot_umap_2d(
    Z_cs_scibert_umap,
    scibert_cs_predictions["KMeans"],
    "SciBERT + UMAP | Domain A / CS by KMeans cluster",
    os.path.join(SCIBERT_STEP7_DIR, "step7_scibert_umap_cs_by_kmeans_cluster.png")
)

# ============================================================
# 7.5 Final Step 7 summary
# ============================================================

step7_summary = pd.concat(
    [tfidf_cs_metrics, scibert_cs_metrics],
    ignore_index=True
)

print("\n========== FINAL STEP 7 SUMMARY ==========")
display(step7_summary)

step7_summary_path = os.path.join(STEP7_OUTPUT_DIR, "step7_domain_a_umap_clustering_summary.csv")
step7_summary.to_csv(step7_summary_path, index=False, encoding="utf-8-sig")

print("\nSaved Step 7 summary:")
print(step7_summary_path)

In [ ]:
# ============================================================
# STEP 8: Apply SAME CS-learned feature setup to Medical Domain
# TF-IDF: use CS-fitted vectorizer.transform()
# SciBERT: use same SciBERT model + CS-fitted scaler
# ============================================================

import os
import numpy as np
import pandas as pd
import joblib
from tqdm.auto import tqdm
from scipy import sparse

import torch
from transformers import AutoTokenizer, AutoModel

STEP8_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "step8_cross_domain_transfer_to_medical")
TFIDF_STEP8_DIR = os.path.join(STEP8_OUTPUT_DIR, "tfidf")
SCIBERT_STEP8_DIR = os.path.join(STEP8_OUTPUT_DIR, "scibert")

os.makedirs(STEP8_OUTPUT_DIR, exist_ok=True)
os.makedirs(TFIDF_STEP8_DIR, exist_ok=True)
os.makedirs(SCIBERT_STEP8_DIR, exist_ok=True)

# ============================================================
# 8.1 Select Domain B = Medical only
# ============================================================

medical_docs = corpus_df[
    (corpus_df["domain"] == "Medical") &
    (corpus_df["text_clean"].str.strip() != "") &
    (corpus_df["preprocessed_text"].str.strip() != "")
].copy().reset_index(drop=True)

medical_ai_mask = medical_docs["label"].values == "AI"
medical_human_mask = medical_docs["label"].values == "Human"

print("========== STEP 8A: DOMAIN B / MEDICAL SELECTION ==========")
print("Medical documents:", len(medical_docs))
print("CS documents used in Step 8:", int((medical_docs["domain"] == "CS").sum()))

print("\nMedical label distribution:")
display(medical_docs["label"].value_counts().reset_index())

print("\nMedical AI documents:", int(medical_ai_mask.sum()))
print("Medical Human documents:", int(medical_human_mask.sum()))

# Save Medical-only documents
medical_docs_path = os.path.join(STEP8_OUTPUT_DIR, "step8_domain_b_medical_documents.csv")
medical_docs.to_csv(medical_docs_path, index=False, encoding="utf-8-sig")

print("\nSaved Medical-only document file:")
print(medical_docs_path)

# ============================================================
# 8.2 Apply SAME CS-fitted TF-IDF vectorizer to Medical
# Important: transform(), not fit_transform()
# ============================================================

X_medical_tfidf_full = tfidf_vectorizer_cs.transform(
    medical_docs["preprocessed_text"]
)

print("\n========== STEP 8B: APPLY CS-FITTED TF-IDF TO MEDICAL ==========")
print("Medical full TF-IDF matrix shape:", X_medical_tfidf_full.shape)
print("CS TF-IDF feature count:", X_cs_tfidf.shape[1])
print("Medical TF-IDF feature count:", X_medical_tfidf_full.shape[1])
print("Feature count matches CS:", X_medical_tfidf_full.shape[1] == X_cs_tfidf.shape[1])
print("Non-zero values:", X_medical_tfidf_full.nnz)
print("Sparsity:", round(1 - (X_medical_tfidf_full.nnz / (X_medical_tfidf_full.shape[0] * X_medical_tfidf_full.shape[1])), 6))
print("All-zero rows in full Medical TF-IDF:", int((np.diff(X_medical_tfidf_full.indptr) == 0).sum()))

# Save full Medical TF-IDF
medical_tfidf_full_path = os.path.join(TFIDF_STEP8_DIR, "step8_X_medical_tfidf_full_using_cs_vectorizer.npz")
sparse.save_npz(medical_tfidf_full_path, X_medical_tfidf_full)

print("\nSaved Medical full TF-IDF matrix:")
print(medical_tfidf_full_path)

# ============================================================
# 8.3 SciBERT embeddings for Medical
# Same SciBERT model as CS
# Same CS-fitted scaler
# ============================================================

SCIBERT_MODEL_NAME = "allenai/scibert_scivocab_uncased"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("\n========== STEP 8C: SCIBERT MEDICAL SETUP ==========")
print("Using device:", device)

if str(device) == "cpu":
    print("WARNING: You are using CPU. SciBERT embedding will be slow. GPU runtime is recommended.")

# If tokenizer/model are not available in memory, reload them
try:
    tokenizer
    scibert_model
    print("Using existing SciBERT tokenizer and model from memory.")
except NameError:
    print("Reloading SciBERT tokenizer and model.")
    tokenizer = AutoTokenizer.from_pretrained(SCIBERT_MODEL_NAME)
    scibert_model = AutoModel.from_pretrained(SCIBERT_MODEL_NAME).to(device)
    scibert_model.eval()

# If scaler is not available in memory, reload it
try:
    scibert_scaler_cs
    print("Using existing CS-fitted SciBERT scaler from memory.")
except NameError:
    print("Reloading CS-fitted SciBERT scaler.")
    scibert_scaler_path = os.path.join(
        SCIBERT_OUTPUT_DIR,
        "step3_scibert_scaler_fitted_on_cs_only.joblib"
    )
    scibert_scaler_cs = joblib.load(scibert_scaler_path)

def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts

def embed_texts_scibert(texts, batch_size=16, max_length=512):
    all_embeddings = []
    texts = list(texts)

    for start in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[start:start + batch_size]

        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        encoded = {k: v.to(device) for k, v in encoded.items()}

        with torch.no_grad():
            outputs = scibert_model(**encoded)
            embeddings = mean_pooling(outputs.last_hidden_state, encoded["attention_mask"])

        all_embeddings.append(embeddings.cpu().numpy())

    return np.vstack(all_embeddings).astype(np.float32)

medical_scibert_raw_path = os.path.join(
    SCIBERT_STEP8_DIR,
    "step8_medical_scibert_raw_embeddings.npy"
)

medical_scibert_scaled_path = os.path.join(
    SCIBERT_STEP8_DIR,
    "step8_medical_scibert_scaled_using_cs_scaler.npy"
)

if os.path.exists(medical_scibert_raw_path):
    E_medical_scibert_raw = np.load(medical_scibert_raw_path)
    print("\nLoaded existing Medical SciBERT raw embeddings:")
    print(medical_scibert_raw_path)
else:
    E_medical_scibert_raw = embed_texts_scibert(
        medical_docs["text_clean"].fillna("").astype(str).values,
        batch_size=16,
        max_length=512
    )

    np.save(medical_scibert_raw_path, E_medical_scibert_raw)

    print("\nSaved new Medical SciBERT raw embeddings:")
    print(medical_scibert_raw_path)

# Apply CS-fitted scaler to Medical
E_medical_scibert = scibert_scaler_cs.transform(E_medical_scibert_raw).astype(np.float32)

np.save(medical_scibert_scaled_path, E_medical_scibert)

print("\n========== STEP 8D: MEDICAL SCIBERT TRANSFER COMPLETE ==========")
print("Medical SciBERT raw embedding shape:", E_medical_scibert_raw.shape)
print("Medical SciBERT scaled embedding shape:", E_medical_scibert.shape)
print("CS SciBERT scaled embedding shape:", E_cs_scibert.shape)
print("SciBERT dimension count matches CS:", E_medical_scibert.shape[1] == E_cs_scibert.shape[1])
print("NaN values:", int(np.isnan(E_medical_scibert).sum()))
print("Infinite values:", int(np.isinf(E_medical_scibert).sum()))

print("\nSaved Medical SciBERT outputs:")
print(medical_scibert_raw_path)
print(medical_scibert_scaled_path)

# ============================================================
# 8.4 Step 8 transfer check summary
# ============================================================

step8_summary = pd.DataFrame([
    {
        "representation": "TF-IDF",
        "domain": "Domain B / Medical",
        "documents": len(medical_docs),
        "ai_documents": int(medical_ai_mask.sum()),
        "human_documents": int(medical_human_mask.sum()),
        "cs_documents_used": int((medical_docs["domain"] == "CS").sum()),
        "matrix_shape": str(X_medical_tfidf_full.shape),
        "feature_or_dimension_count": X_medical_tfidf_full.shape[1],
        "matches_cs_feature_count": X_medical_tfidf_full.shape[1] == X_cs_tfidf.shape[1],
        "all_zero_rows": int((np.diff(X_medical_tfidf_full.indptr) == 0).sum())
    },
    {
        "representation": "SciBERT",
        "domain": "Domain B / Medical",
        "documents": len(medical_docs),
        "ai_documents": int(medical_ai_mask.sum()),
        "human_documents": int(medical_human_mask.sum()),
        "cs_documents_used": int((medical_docs["domain"] == "CS").sum()),
        "matrix_shape": str(E_medical_scibert.shape),
        "feature_or_dimension_count": E_medical_scibert.shape[1],
        "matches_cs_feature_count": E_medical_scibert.shape[1] == E_cs_scibert.shape[1],
        "all_zero_rows": int((np.linalg.norm(E_medical_scibert, axis=1) == 0).sum())
    }
])

print("\n========== FINAL STEP 8 SUMMARY ==========")
display(step8_summary)

step8_summary_path = os.path.join(STEP8_OUTPUT_DIR, "step8_cross_domain_transfer_summary.csv")
step8_summary.to_csv(step8_summary_path, index=False, encoding="utf-8-sig")

print("\nSaved Step 8 summary:")
print(step8_summary_path)

In [ ]:
# ============================================================
# STEP 9: Domain B / Medical Feature Matrix Construction
# TF-IDF DTM using SAME CS-selected 200 contrast vocabulary
# SciBERT matrix using SAME CS-selected 200 contrast dimensions
# ============================================================

import os
import numpy as np
import pandas as pd

from scipy import sparse
from sklearn.preprocessing import normalize

STEP9_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "step9_domain_b_feature_matrix")
TFIDF_STEP9_DIR = os.path.join(STEP9_OUTPUT_DIR, "tfidf")
SCIBERT_STEP9_DIR = os.path.join(STEP9_OUTPUT_DIR, "scibert")

os.makedirs(STEP9_OUTPUT_DIR, exist_ok=True)
os.makedirs(TFIDF_STEP9_DIR, exist_ok=True)
os.makedirs(SCIBERT_STEP9_DIR, exist_ok=True)

# ============================================================
# 9.1 Safety checks
# ============================================================

print("========== STEP 9 SAFETY CHECK ==========")

print("Medical documents:", len(medical_docs))
print("Medical full TF-IDF matrix shape:", X_medical_tfidf_full.shape)
print("Medical full SciBERT matrix shape:", E_medical_scibert.shape)

print("\nSelected TF-IDF feature IDs:", len(selected_tfidf_feature_ids))
print("Selected TF-IDF terms:", len(selected_tfidf_terms))
print("Selected SciBERT dimension IDs:", len(selected_scibert_dim_ids))

assert len(medical_docs) == X_medical_tfidf_full.shape[0], "Medical docs and TF-IDF rows do not match."
assert len(medical_docs) == E_medical_scibert.shape[0], "Medical docs and SciBERT rows do not match."
assert len(selected_tfidf_feature_ids) == 200, "TF-IDF selected vocabulary should be 200."
assert len(selected_scibert_dim_ids) == 200, "SciBERT selected dimensions should be 200."
assert int((medical_docs["domain"] == "CS").sum()) == 0, "CS documents found in Step 9 Medical data."

medical_ai_mask = medical_docs["label"].values == "AI"
medical_human_mask = medical_docs["label"].values == "Human"

print("\nMedical AI documents:", int(medical_ai_mask.sum()))
print("Medical Human documents:", int(medical_human_mask.sum()))

# ============================================================
# 9.2 TF-IDF DTM construction for Domain B / Medical
# Select same 200 CS-learned contrast-vocabulary columns
# ============================================================

X_medical_tfidf_dtm = X_medical_tfidf_full[:, selected_tfidf_feature_ids]

# Normalize selected DTM before UMAP/clustering
X_medical_tfidf_dtm_norm = normalize(X_medical_tfidf_dtm, norm="l2", axis=1)

tfidf_med_nonzero_per_doc = np.diff(X_medical_tfidf_dtm.indptr)
tfidf_med_nonzero_ai = tfidf_med_nonzero_per_doc[medical_ai_mask]
tfidf_med_nonzero_human = tfidf_med_nonzero_per_doc[medical_human_mask]

print("\n========== STEP 9A: TF-IDF DTM DOMAIN B / MEDICAL ==========")
print("DTM matrix shape:", X_medical_tfidf_dtm.shape)
print("Rows/documents:", X_medical_tfidf_dtm.shape[0])
print("Vocabulary terms:", X_medical_tfidf_dtm.shape[1])
print("Non-zero DTM values:", X_medical_tfidf_dtm.nnz)
print("Sparsity:", round(1 - (X_medical_tfidf_dtm.nnz / (X_medical_tfidf_dtm.shape[0] * X_medical_tfidf_dtm.shape[1])), 6))
print("All-zero rows:", int((tfidf_med_nonzero_per_doc == 0).sum()))
print("AI avg non-zero terms/doc:", round(float(tfidf_med_nonzero_ai.mean()), 4))
print("Human avg non-zero terms/doc:", round(float(tfidf_med_nonzero_human.mean()), 4))
print("AI all-zero rows:", int((tfidf_med_nonzero_ai == 0).sum()))
print("Human all-zero rows:", int((tfidf_med_nonzero_human == 0).sum()))

# Identify any all-zero Medical rows under selected vocabulary
all_zero_medical_idx = np.where(tfidf_med_nonzero_per_doc == 0)[0]

if len(all_zero_medical_idx) > 0:
    print("\nMedical documents with zero selected TF-IDF vocabulary terms:")
    display(
        medical_docs.iloc[all_zero_medical_idx][
            ["doc_id", "domain", "label", "title", "clean_word_count", "token_count"]
        ].head(20)
    )
else:
    print("\nNo all-zero Medical rows under selected TF-IDF vocabulary.")

print("\nSelected TF-IDF vocabulary preview:")
display(
    selected_tfidf_vocab[
        ["vocab_group", "feature_id", "feature", "contrast_ai_minus_human"]
    ].head(10)
)

# Save TF-IDF Step 9 outputs
medical_tfidf_dtm_path = os.path.join(TFIDF_STEP9_DIR, "step9_X_medical_tfidf_200_dtm.npz")
medical_tfidf_dtm_norm_path = os.path.join(TFIDF_STEP9_DIR, "step9_X_medical_tfidf_200_dtm_l2_normalized.npz")
medical_tfidf_vocab_path = os.path.join(TFIDF_STEP9_DIR, "step9_selected_tfidf_vocabulary_used.csv")

sparse.save_npz(medical_tfidf_dtm_path, X_medical_tfidf_dtm)
sparse.save_npz(medical_tfidf_dtm_norm_path, X_medical_tfidf_dtm_norm)
selected_tfidf_vocab.to_csv(medical_tfidf_vocab_path, index=False, encoding="utf-8-sig")

print("\nSaved TF-IDF Step 9 outputs:")
print(medical_tfidf_dtm_path)
print(medical_tfidf_dtm_norm_path)
print(medical_tfidf_vocab_path)

# ============================================================
# 9.3 SciBERT selected feature matrix for Domain B / Medical
# Select same 200 CS-derived contrast dimensions
# ============================================================

X_medical_scibert_selected = E_medical_scibert[:, selected_scibert_dim_ids]

# Normalize selected SciBERT dimensions before UMAP/clustering
X_medical_scibert_selected_norm = normalize(X_medical_scibert_selected, norm="l2", axis=1)

print("\n========== STEP 9B: SCIBERT FEATURE MATRIX DOMAIN B / MEDICAL ==========")
print("Selected SciBERT matrix shape:", X_medical_scibert_selected.shape)
print("Rows/documents:", X_medical_scibert_selected.shape[0])
print("Selected dimensions:", X_medical_scibert_selected.shape[1])
print("NaN values:", int(np.isnan(X_medical_scibert_selected).sum()))
print("Infinite values:", int(np.isinf(X_medical_scibert_selected).sum()))
print("Minimum value:", float(np.min(X_medical_scibert_selected)))
print("Maximum value:", float(np.max(X_medical_scibert_selected)))
print("Mean value:", float(np.mean(X_medical_scibert_selected)))
print("Std value:", float(np.std(X_medical_scibert_selected)))
print("All-zero rows:", int((np.linalg.norm(X_medical_scibert_selected, axis=1) == 0).sum()))

print("\nSelected SciBERT dimensions preview:")
display(
    selected_scibert_features[
        ["feature_group", "dimension_id", "contrast_ai_minus_human"]
    ].head(10)
)

# Save SciBERT Step 9 outputs
medical_scibert_selected_path = os.path.join(
    SCIBERT_STEP9_DIR,
    "step9_X_medical_scibert_200_selected_dimensions.npy"
)

medical_scibert_selected_norm_path = os.path.join(
    SCIBERT_STEP9_DIR,
    "step9_X_medical_scibert_200_selected_dimensions_l2_normalized.npy"
)

medical_scibert_dims_path = os.path.join(
    SCIBERT_STEP9_DIR,
    "step9_selected_scibert_dimensions_used.csv"
)

np.save(medical_scibert_selected_path, X_medical_scibert_selected)
np.save(medical_scibert_selected_norm_path, X_medical_scibert_selected_norm)
selected_scibert_features.to_csv(medical_scibert_dims_path, index=False, encoding="utf-8-sig")

print("\nSaved SciBERT Step 9 outputs:")
print(medical_scibert_selected_path)
print(medical_scibert_selected_norm_path)
print(medical_scibert_dims_path)

# ============================================================
# 9.4 Final Step 9 summary
# ============================================================

step9_summary = pd.DataFrame([
    {
        "representation": "TF-IDF",
        "domain": "Domain B / Medical",
        "matrix_shape": str(X_medical_tfidf_dtm.shape),
        "documents": X_medical_tfidf_dtm.shape[0],
        "features_or_dimensions": X_medical_tfidf_dtm.shape[1],
        "nonzero_values": X_medical_tfidf_dtm.nnz,
        "sparsity": round(1 - (X_medical_tfidf_dtm.nnz / (X_medical_tfidf_dtm.shape[0] * X_medical_tfidf_dtm.shape[1])), 6),
        "all_zero_rows": int((tfidf_med_nonzero_per_doc == 0).sum()),
        "ai_avg_nonzero": round(float(tfidf_med_nonzero_ai.mean()), 4),
        "human_avg_nonzero": round(float(tfidf_med_nonzero_human.mean()), 4),
        "ai_all_zero_rows": int((tfidf_med_nonzero_ai == 0).sum()),
        "human_all_zero_rows": int((tfidf_med_nonzero_human == 0).sum())
    },
    {
        "representation": "SciBERT",
        "domain": "Domain B / Medical",
        "matrix_shape": str(X_medical_scibert_selected.shape),
        "documents": X_medical_scibert_selected.shape[0],
        "features_or_dimensions": X_medical_scibert_selected.shape[1],
        "nonzero_values": "Dense matrix",
        "sparsity": "Not applicable",
        "all_zero_rows": int((np.linalg.norm(X_medical_scibert_selected, axis=1) == 0).sum()),
        "ai_avg_nonzero": "Not applicable",
        "human_avg_nonzero": "Not applicable",
        "ai_all_zero_rows": "Not applicable",
        "human_all_zero_rows": "Not applicable"
    }
])

print("\n========== FINAL STEP 9 SUMMARY ==========")
display(step9_summary)

step9_summary_path = os.path.join(STEP9_OUTPUT_DIR, "step9_domain_b_feature_matrix_summary.csv")
step9_summary.to_csv(step9_summary_path, index=False, encoding="utf-8-sig")

print("\nSaved Step 9 summary:")
print(step9_summary_path)

In [ ]:
# ============================================================
# STEP 10: UMAP + Clustering on Domain B / Medical
# IMPORTANT:
# Use SAME UMAP reducers fitted on CS in Step 7.
# Do NOT fit UMAP again on Medical.
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score

STEP10_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "step10_domain_b_umap_clustering")
TFIDF_STEP10_DIR = os.path.join(STEP10_OUTPUT_DIR, "tfidf")
SCIBERT_STEP10_DIR = os.path.join(STEP10_OUTPUT_DIR, "scibert")

os.makedirs(STEP10_OUTPUT_DIR, exist_ok=True)
os.makedirs(TFIDF_STEP10_DIR, exist_ok=True)
os.makedirs(SCIBERT_STEP10_DIR, exist_ok=True)

# ============================================================
# 10.1 Helper functions
# These are redefined here in case Colab memory was reset
# ============================================================

def label_to_int(labels):
    return np.array([1 if x == "AI" else 0 for x in labels])

def purity_score(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    total_correct = 0

    for cluster_id in np.unique(y_pred):
        idx = y_pred == cluster_id
        cluster_true_labels = y_true[idx]

        if len(cluster_true_labels) == 0:
            continue

        _, counts = np.unique(cluster_true_labels, return_counts=True)
        total_correct += counts.max()

    return total_correct / len(y_true)

def cluster_composition_table(true_labels, pred_labels):
    table = pd.crosstab(
        pd.Series(pred_labels, name="cluster"),
        pd.Series(true_labels, name="true_label")
    )

    rows = []

    for cluster_id in sorted(np.unique(pred_labels)):
        idx = pred_labels == cluster_id
        cluster_labels = np.asarray(true_labels)[idx]

        values, counts = np.unique(cluster_labels, return_counts=True)
        majority_label = values[np.argmax(counts)]
        cluster_purity = counts.max() / counts.sum()

        rows.append({
            "cluster": cluster_id,
            "cluster_size": int(idx.sum()),
            "majority_label": majority_label,
            "cluster_purity": cluster_purity
        })

    summary = pd.DataFrame(rows)
    return table, summary

def evaluate_clustering(X, true_labels, pred_labels, representation_name, domain_name, algorithm_name):
    y_true_int = label_to_int(true_labels)

    purity = purity_score(y_true_int, pred_labels)
    ari = adjusted_rand_score(y_true_int, pred_labels)
    nmi = normalized_mutual_info_score(y_true_int, pred_labels)

    try:
        sil = silhouette_score(X, pred_labels)
    except Exception:
        sil = np.nan

    return {
        "representation": representation_name,
        "domain": domain_name,
        "algorithm": algorithm_name,
        "documents": len(true_labels),
        "human_docs": int((np.asarray(true_labels) == "Human").sum()),
        "ai_docs": int((np.asarray(true_labels) == "AI").sum()),
        "purity": purity,
        "ari": ari,
        "nmi": nmi,
        "silhouette": sil
    }

def run_clustering_suite(X, true_labels, representation_name, domain_name):
    results = []
    predictions = {}

    # 1. KMeans
    kmeans = KMeans(
        n_clusters=2,
        random_state=RANDOM_STATE,
        n_init=30
    )
    pred_kmeans = kmeans.fit_predict(X)
    predictions["KMeans"] = pred_kmeans
    results.append(
        evaluate_clustering(
            X, true_labels, pred_kmeans,
            representation_name, domain_name, "KMeans"
        )
    )

    # 2. Agglomerative
    agg = AgglomerativeClustering(
        n_clusters=2,
        linkage="ward"
    )
    pred_agg = agg.fit_predict(X)
    predictions["Agglomerative"] = pred_agg
    results.append(
        evaluate_clustering(
            X, true_labels, pred_agg,
            representation_name, domain_name, "Agglomerative"
        )
    )

    # 3. Gaussian Mixture
    gmm = GaussianMixture(
        n_components=2,
        random_state=RANDOM_STATE,
        covariance_type="full"
    )
    pred_gmm = gmm.fit_predict(X)
    predictions["GMM"] = pred_gmm
    results.append(
        evaluate_clustering(
            X, true_labels, pred_gmm,
            representation_name, domain_name, "GMM"
        )
    )

    metrics_df = pd.DataFrame(results)
    return metrics_df, predictions

def plot_umap_2d(Z, labels, title, save_path):
    plt.figure(figsize=(7, 5))

    labels = np.asarray(labels)

    for lab in np.unique(labels):
        idx = labels == lab
        plt.scatter(
            Z[idx, 0],
            Z[idx, 1],
            s=8,
            alpha=0.65,
            label=str(lab)
        )

    plt.title(title)
    plt.xlabel("UMAP-1")
    plt.ylabel("UMAP-2")
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.show()

# ============================================================
# 10.2 Safety check
# ============================================================

print("========== STEP 10 SAFETY CHECK ==========")

print("Medical documents:", len(medical_docs))
print("TF-IDF Medical selected normalized matrix shape:", X_medical_tfidf_dtm_norm.shape)
print("SciBERT Medical selected normalized matrix shape:", X_medical_scibert_selected_norm.shape)

assert X_medical_tfidf_dtm_norm.shape == (3990, 200), "TF-IDF Medical DTM should be (3990, 200)."
assert X_medical_scibert_selected_norm.shape == (3990, 200), "SciBERT Medical matrix should be (3990, 200)."

true_labels_medical = medical_docs["label"].values

print("\nTrue Medical label distribution:")
display(pd.Series(true_labels_medical).value_counts().reset_index())

# If UMAP reducers are not in memory, reload them from Step 7
try:
    tfidf_umap_reducer
    print("\nUsing existing TF-IDF UMAP reducer from memory.")
except NameError:
    print("\nReloading TF-IDF UMAP reducer fitted on CS.")
    tfidf_umap_reducer_path = os.path.join(
        OUTPUT_DIR,
        "step7_domain_a_umap_clustering",
        "tfidf",
        "step7_tfidf_umap_reducer_fitted_on_cs.joblib"
    )
    tfidf_umap_reducer = joblib.load(tfidf_umap_reducer_path)

try:
    scibert_umap_reducer
    print("Using existing SciBERT UMAP reducer from memory.")
except NameError:
    print("Reloading SciBERT UMAP reducer fitted on CS.")
    scibert_umap_reducer_path = os.path.join(
        OUTPUT_DIR,
        "step7_domain_a_umap_clustering",
        "scibert",
        "step7_scibert_umap_reducer_fitted_on_cs.joblib"
    )
    scibert_umap_reducer = joblib.load(scibert_umap_reducer_path)

# ============================================================
# 10.3 TF-IDF + UMAP transform + clustering on Medical
# ============================================================

print("\n========== STEP 10A: TF-IDF + UMAP DOMAIN B / MEDICAL ==========")

Z_medical_tfidf_umap = tfidf_umap_reducer.transform(X_medical_tfidf_dtm_norm)

print("Original Medical TF-IDF DTM shape:", X_medical_tfidf_dtm_norm.shape)
print("UMAP-reduced Medical TF-IDF shape:", Z_medical_tfidf_umap.shape)
print("NaN values:", int(np.isnan(Z_medical_tfidf_umap).sum()))
print("Infinite values:", int(np.isinf(Z_medical_tfidf_umap).sum()))

tfidf_medical_metrics, tfidf_medical_predictions = run_clustering_suite(
    Z_medical_tfidf_umap,
    true_labels_medical,
    representation_name="TF-IDF + UMAP",
    domain_name="Domain B / Medical"
)

print("\nTF-IDF + UMAP Medical clustering metrics:")
display(tfidf_medical_metrics)

for algo_name, pred in tfidf_medical_predictions.items():
    print(f"\nTF-IDF + UMAP | Medical | {algo_name} cluster composition:")
    comp_table, comp_summary = cluster_composition_table(true_labels_medical, pred)
    display(comp_table)
    display(comp_summary)

# Save TF-IDF Step 10 outputs
tfidf_medical_umap_matrix_path = os.path.join(TFIDF_STEP10_DIR, "step10_Z_medical_tfidf_umap.npy")
tfidf_medical_metrics_path = os.path.join(TFIDF_STEP10_DIR, "step10_tfidf_medical_clustering_metrics.csv")

np.save(tfidf_medical_umap_matrix_path, Z_medical_tfidf_umap)
tfidf_medical_metrics.to_csv(tfidf_medical_metrics_path, index=False, encoding="utf-8-sig")

for algo_name, pred in tfidf_medical_predictions.items():
    pred_path = os.path.join(TFIDF_STEP10_DIR, f"step10_tfidf_medical_{algo_name}_clusters.csv")

    pred_df = medical_docs[["doc_id", "domain", "label", "title"]].copy()
    pred_df["cluster"] = pred

    pred_df.to_csv(pred_path, index=False, encoding="utf-8-sig")

print("\nSaved TF-IDF Step 10 outputs:")
print(tfidf_medical_umap_matrix_path)
print(tfidf_medical_metrics_path)

plot_umap_2d(
    Z_medical_tfidf_umap,
    true_labels_medical,
    "TF-IDF + UMAP | Domain B / Medical by true label",
    os.path.join(TFIDF_STEP10_DIR, "step10_tfidf_umap_medical_by_true_label.png")
)

plot_umap_2d(
    Z_medical_tfidf_umap,
    tfidf_medical_predictions["KMeans"],
    "TF-IDF + UMAP | Domain B / Medical by KMeans cluster",
    os.path.join(TFIDF_STEP10_DIR, "step10_tfidf_umap_medical_by_kmeans_cluster.png")
)

# ============================================================
# 10.4 SciBERT + UMAP transform + clustering on Medical
# ============================================================

print("\n========== STEP 10B: SCIBERT + UMAP DOMAIN B / MEDICAL ==========")

Z_medical_scibert_umap = scibert_umap_reducer.transform(X_medical_scibert_selected_norm)

print("Original Medical SciBERT selected matrix shape:", X_medical_scibert_selected_norm.shape)
print("UMAP-reduced Medical SciBERT shape:", Z_medical_scibert_umap.shape)
print("NaN values:", int(np.isnan(Z_medical_scibert_umap).sum()))
print("Infinite values:", int(np.isinf(Z_medical_scibert_umap).sum()))

scibert_medical_metrics, scibert_medical_predictions = run_clustering_suite(
    Z_medical_scibert_umap,
    true_labels_medical,
    representation_name="SciBERT + UMAP",
    domain_name="Domain B / Medical"
)

print("\nSciBERT + UMAP Medical clustering metrics:")
display(scibert_medical_metrics)

for algo_name, pred in scibert_medical_predictions.items():
    print(f"\nSciBERT + UMAP | Medical | {algo_name} cluster composition:")
    comp_table, comp_summary = cluster_composition_table(true_labels_medical, pred)
    display(comp_table)
    display(comp_summary)

# Save SciBERT Step 10 outputs
scibert_medical_umap_matrix_path = os.path.join(SCIBERT_STEP10_DIR, "step10_Z_medical_scibert_umap.npy")
scibert_medical_metrics_path = os.path.join(SCIBERT_STEP10_DIR, "step10_scibert_medical_clustering_metrics.csv")

np.save(scibert_medical_umap_matrix_path, Z_medical_scibert_umap)
scibert_medical_metrics.to_csv(scibert_medical_metrics_path, index=False, encoding="utf-8-sig")

for algo_name, pred in scibert_medical_predictions.items():
    pred_path = os.path.join(SCIBERT_STEP10_DIR, f"step10_scibert_medical_{algo_name}_clusters.csv")

    pred_df = medical_docs[["doc_id", "domain", "label", "title"]].copy()
    pred_df["cluster"] = pred

    pred_df.to_csv(pred_path, index=False, encoding="utf-8-sig")

print("\nSaved SciBERT Step 10 outputs:")
print(scibert_medical_umap_matrix_path)
print(scibert_medical_metrics_path)

plot_umap_2d(
    Z_medical_scibert_umap,
    true_labels_medical,
    "SciBERT + UMAP | Domain B / Medical by true label",
    os.path.join(SCIBERT_STEP10_DIR, "step10_scibert_umap_medical_by_true_label.png")
)

plot_umap_2d(
    Z_medical_scibert_umap,
    scibert_medical_predictions["KMeans"],
    "SciBERT + UMAP | Domain B / Medical by KMeans cluster",
    os.path.join(SCIBERT_STEP10_DIR, "step10_scibert_umap_medical_by_kmeans_cluster.png")
)

# ============================================================
# 10.5 Final Step 10 summary
# ============================================================

step10_summary = pd.concat(
    [tfidf_medical_metrics, scibert_medical_metrics],
    ignore_index=True
)

print("\n========== FINAL STEP 10 SUMMARY ==========")
display(step10_summary)

step10_summary_path = os.path.join(STEP10_OUTPUT_DIR, "step10_domain_b_umap_clustering_summary.csv")
step10_summary.to_csv(step10_summary_path, index=False, encoding="utf-8-sig")

print("\nSaved Step 10 summary:")
print(step10_summary_path)

In [ ]:
# ============================================================
# STEP 11: Evaluation
# Purity | ARI | NMI | Coherence | Jaccard | UMAP-based stability
# ============================================================

import os
import numpy as np
import pandas as pd
from scipy import sparse

STEP11_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "step11_evaluation")
TFIDF_STEP11_DIR = os.path.join(STEP11_OUTPUT_DIR, "tfidf")
SCIBERT_STEP11_DIR = os.path.join(STEP11_OUTPUT_DIR, "scibert")

os.makedirs(STEP11_OUTPUT_DIR, exist_ok=True)
os.makedirs(TFIDF_STEP11_DIR, exist_ok=True)
os.makedirs(SCIBERT_STEP11_DIR, exist_ok=True)

# ============================================================
# 11.1 Final clustering metric summary
# ============================================================

print("========== STEP 11A: FINAL CLUSTERING METRICS ==========")

all_clustering_metrics = pd.concat(
    [
        tfidf_cs_metrics,
        tfidf_medical_metrics,
        scibert_cs_metrics,
        scibert_medical_metrics
    ],
    ignore_index=True
)

display(all_clustering_metrics)

kmeans_summary = all_clustering_metrics[
    all_clustering_metrics["algorithm"] == "KMeans"
].copy().reset_index(drop=True)

print("\n========== KMEANS-ONLY SUMMARY ==========")
display(kmeans_summary)

all_clustering_metrics_path = os.path.join(
    STEP11_OUTPUT_DIR,
    "step11_all_clustering_metrics.csv"
)

kmeans_summary_path = os.path.join(
    STEP11_OUTPUT_DIR,
    "step11_kmeans_only_summary.csv"
)

all_clustering_metrics.to_csv(all_clustering_metrics_path, index=False, encoding="utf-8-sig")
kmeans_summary.to_csv(kmeans_summary_path, index=False, encoding="utf-8-sig")

print("\nSaved clustering metric outputs:")
print(all_clustering_metrics_path)
print(kmeans_summary_path)

# ============================================================
# 11.2 Helper: Jaccard
# ============================================================

def jaccard_score_sets(a, b):
    a = set(a)
    b = set(b)

    union = a.union(b)

    if len(union) == 0:
        return np.nan

    return len(a.intersection(b)) / len(union)

# ============================================================
# 11.3 TF-IDF cross-domain vocabulary presence and direction
# ============================================================

print("\n========== STEP 11B: TF-IDF VOCABULARY STABILITY ==========")

# Medical contrast using same selected 200 CS vocabulary
X_medical_ai_selected = X_medical_tfidf_dtm[medical_ai_mask]
X_medical_human_selected = X_medical_tfidf_dtm[medical_human_mask]

mean_medical_tfidf_ai_selected = np.asarray(X_medical_ai_selected.mean(axis=0)).ravel()
mean_medical_tfidf_human_selected = np.asarray(X_medical_human_selected.mean(axis=0)).ravel()

medical_tfidf_contrast_selected = (
    mean_medical_tfidf_ai_selected - mean_medical_tfidf_human_selected
)

tfidf_stability_df = selected_tfidf_vocab.copy().reset_index(drop=True)

tfidf_stability_df["medical_mean_tfidf_ai"] = mean_medical_tfidf_ai_selected
tfidf_stability_df["medical_mean_tfidf_human"] = mean_medical_tfidf_human_selected
tfidf_stability_df["medical_contrast_ai_minus_human"] = medical_tfidf_contrast_selected

tfidf_stability_df["cs_sign"] = np.sign(tfidf_stability_df["contrast_ai_minus_human"])
tfidf_stability_df["medical_sign"] = np.sign(tfidf_stability_df["medical_contrast_ai_minus_human"])
tfidf_stability_df["same_direction"] = (
    tfidf_stability_df["cs_sign"] == tfidf_stability_df["medical_sign"]
)

# Presence Jaccard: selected CS terms that appear in Medical selected DTM
medical_selected_term_presence = np.asarray(
    (X_medical_tfidf_dtm > 0).sum(axis=0)
).ravel() > 0

cs_selected_terms = set(tfidf_stability_df["feature"])
medical_present_terms = set(tfidf_stability_df.loc[medical_selected_term_presence, "feature"])

presence_jaccard_all = jaccard_score_sets(cs_selected_terms, medical_present_terms)

cs_ai_terms = set(
    tfidf_stability_df.loc[tfidf_stability_df["vocab_group"] == "AI", "feature"]
)

cs_human_terms = set(
    tfidf_stability_df.loc[tfidf_stability_df["vocab_group"] == "Human", "feature"]
)

medical_ai_direction_terms = set(
    tfidf_stability_df.loc[
        tfidf_stability_df["medical_contrast_ai_minus_human"] > 0,
        "feature"
    ]
)

medical_human_direction_terms = set(
    tfidf_stability_df.loc[
        tfidf_stability_df["medical_contrast_ai_minus_human"] < 0,
        "feature"
    ]
)

directional_jaccard_ai = jaccard_score_sets(
    cs_ai_terms,
    medical_ai_direction_terms
)

directional_jaccard_human = jaccard_score_sets(
    cs_human_terms,
    medical_human_direction_terms
)

overall_direction_agreement = tfidf_stability_df["same_direction"].mean()

ai_direction_agreement = tfidf_stability_df.loc[
    tfidf_stability_df["vocab_group"] == "AI",
    "same_direction"
].mean()

human_direction_agreement = tfidf_stability_df.loc[
    tfidf_stability_df["vocab_group"] == "Human",
    "same_direction"
].mean()

tfidf_stability_summary = pd.DataFrame([
    {
        "representation": "TF-IDF + UMAP",
        "presence_jaccard_all_terms": presence_jaccard_all,
        "directional_jaccard_ai_terms": directional_jaccard_ai,
        "directional_jaccard_human_terms": directional_jaccard_human,
        "overall_direction_agreement": overall_direction_agreement,
        "ai_vocab_direction_agreement": ai_direction_agreement,
        "human_vocab_direction_agreement": human_direction_agreement,
        "medical_terms_present": len(medical_present_terms),
        "total_selected_terms": len(cs_selected_terms)
    }
])

display(tfidf_stability_summary)

print("\nTF-IDF selected terms with changed direction:")
display(
    tfidf_stability_df[
        tfidf_stability_df["same_direction"] == False
    ][
        [
            "vocab_group",
            "feature",
            "contrast_ai_minus_human",
            "medical_contrast_ai_minus_human",
            "same_direction"
        ]
    ].head(30)
)

tfidf_stability_path = os.path.join(
    TFIDF_STEP11_DIR,
    "step11_tfidf_directional_stability.csv"
)

tfidf_stability_summary_path = os.path.join(
    TFIDF_STEP11_DIR,
    "step11_tfidf_stability_summary.csv"
)

tfidf_stability_df.to_csv(tfidf_stability_path, index=False, encoding="utf-8-sig")
tfidf_stability_summary.to_csv(tfidf_stability_summary_path, index=False, encoding="utf-8-sig")

print("\nSaved TF-IDF stability outputs:")
print(tfidf_stability_path)
print(tfidf_stability_summary_path)

# ============================================================
# 11.4 TF-IDF NPMI vocabulary coherence
# ============================================================

print("\n========== STEP 11C: TF-IDF NPMI COHERENCE ==========")

def npmi_coherence_from_matrix(X, col_positions, eps=1e-12):
    """
    Compute NPMI coherence over selected columns.
    X should be a document-term matrix.
    col_positions are positions inside the selected 200-column matrix.
    """

    if sparse.issparse(X):
        B = (X[:, col_positions] > 0).astype(np.int32).toarray()
    else:
        B = (X[:, col_positions] > 0).astype(np.int32)

    n_docs = B.shape[0]
    n_terms = B.shape[1]

    if n_terms < 2:
        return {
            "mean_npmi": np.nan,
            "median_npmi": np.nan,
            "zero_cooccurrence_pairs": np.nan
        }

    df = B.sum(axis=0)
    co = B.T @ B

    scores = []
    zero_pairs = 0

    for i in range(n_terms):
        for j in range(i + 1, n_terms):
            p_i = df[i] / n_docs
            p_j = df[j] / n_docs
            p_ij = co[i, j] / n_docs

            if p_ij <= 0:
                zero_pairs += 1
                continue

            pmi = np.log((p_ij + eps) / ((p_i * p_j) + eps))
            npmi = pmi / (-np.log(p_ij + eps))

            scores.append(npmi)

    if len(scores) == 0:
        return {
            "mean_npmi": np.nan,
            "median_npmi": np.nan,
            "zero_cooccurrence_pairs": zero_pairs
        }

    return {
        "mean_npmi": float(np.mean(scores)),
        "median_npmi": float(np.median(scores)),
        "zero_cooccurrence_pairs": int(zero_pairs)
    }

ai_positions = list(range(0, 100))
human_positions = list(range(100, 200))
all_positions = list(range(0, 200))

coherence_rows = []

for domain_name, Xmat in [
    ("Domain A / CS", X_cs_tfidf_dtm),
    ("Domain B / Medical", X_medical_tfidf_dtm)
]:
    for vocab_name, positions in [
        ("AI contrast vocab", ai_positions),
        ("Human contrast vocab", human_positions),
        ("All contrast vocab", all_positions)
    ]:
        coherence_result = npmi_coherence_from_matrix(Xmat, positions)

        coherence_rows.append({
            "representation": "TF-IDF",
            "domain": domain_name,
            "vocabulary_group": vocab_name,
            **coherence_result
        })

tfidf_coherence_df = pd.DataFrame(coherence_rows)

display(tfidf_coherence_df)

tfidf_coherence_path = os.path.join(
    TFIDF_STEP11_DIR,
    "step11_tfidf_npmi_coherence.csv"
)

tfidf_coherence_df.to_csv(tfidf_coherence_path, index=False, encoding="utf-8-sig")

print("\nSaved TF-IDF coherence output:")
print(tfidf_coherence_path)

# ============================================================
# 11.5 SciBERT cross-domain dimension stability
# ============================================================

print("\n========== STEP 11D: SCIBERT DIMENSION STABILITY ==========")

# Medical contrast using all 768 SciBERT dimensions
E_medical_scibert_ai = E_medical_scibert[medical_ai_mask]
E_medical_scibert_human = E_medical_scibert[medical_human_mask]

mean_medical_scibert_ai = E_medical_scibert_ai.mean(axis=0)
mean_medical_scibert_human = E_medical_scibert_human.mean(axis=0)

medical_scibert_contrast_all = (
    mean_medical_scibert_ai - mean_medical_scibert_human
)

medical_scibert_contrast_df = pd.DataFrame({
    "dimension_id": np.arange(E_medical_scibert.shape[1]),
    "medical_mean_scibert_ai": mean_medical_scibert_ai,
    "medical_mean_scibert_human": mean_medical_scibert_human,
    "medical_contrast_ai_minus_human": medical_scibert_contrast_all,
    "medical_abs_contrast": np.abs(medical_scibert_contrast_all)
})

# Stability only for selected 200 CS dimensions
scibert_stability_df = selected_scibert_features.copy().reset_index(drop=True)

scibert_stability_df["medical_mean_scibert_ai"] = mean_medical_scibert_ai[
    scibert_stability_df["dimension_id"].values
]

scibert_stability_df["medical_mean_scibert_human"] = mean_medical_scibert_human[
    scibert_stability_df["dimension_id"].values
]

scibert_stability_df["medical_contrast_ai_minus_human"] = medical_scibert_contrast_all[
    scibert_stability_df["dimension_id"].values
]

scibert_stability_df["cs_sign"] = np.sign(scibert_stability_df["contrast_ai_minus_human"])
scibert_stability_df["medical_sign"] = np.sign(scibert_stability_df["medical_contrast_ai_minus_human"])

scibert_stability_df["same_direction"] = (
    scibert_stability_df["cs_sign"] == scibert_stability_df["medical_sign"]
)

# Jaccard-style dimension overlap
medical_top_ai_dims = set(
    medical_scibert_contrast_df
    .sort_values("medical_contrast_ai_minus_human", ascending=False)
    .head(100)["dimension_id"]
)

medical_top_human_dims = set(
    medical_scibert_contrast_df
    .sort_values("medical_contrast_ai_minus_human", ascending=True)
    .head(100)["dimension_id"]
)

cs_top_ai_dims = set(top_ai_scibert_dims["dimension_id"])
cs_top_human_dims = set(top_human_scibert_dims["dimension_id"])

dimension_jaccard_ai = jaccard_score_sets(cs_top_ai_dims, medical_top_ai_dims)
dimension_jaccard_human = jaccard_score_sets(cs_top_human_dims, medical_top_human_dims)

overall_scibert_direction_agreement = scibert_stability_df["same_direction"].mean()

ai_scibert_direction_agreement = scibert_stability_df.loc[
    scibert_stability_df["feature_group"] == "AI",
    "same_direction"
].mean()

human_scibert_direction_agreement = scibert_stability_df.loc[
    scibert_stability_df["feature_group"] == "Human",
    "same_direction"
].mean()

scibert_stability_summary = pd.DataFrame([
    {
        "representation": "SciBERT + UMAP",
        "dimension_jaccard_ai": dimension_jaccard_ai,
        "dimension_jaccard_human": dimension_jaccard_human,
        "overall_direction_agreement": overall_scibert_direction_agreement,
        "ai_dimension_direction_agreement": ai_scibert_direction_agreement,
        "human_dimension_direction_agreement": human_scibert_direction_agreement,
        "selected_dimensions": len(scibert_stability_df),
        "coherence_note": "NPMI word coherence is not applicable to SciBERT embedding dimensions."
    }
])

display(scibert_stability_summary)

print("\nSciBERT selected dimensions with changed direction:")
display(
    scibert_stability_df[
        scibert_stability_df["same_direction"] == False
    ][
        [
            "feature_group",
            "dimension_id",
            "contrast_ai_minus_human",
            "medical_contrast_ai_minus_human",
            "same_direction"
        ]
    ].head(30)
)

scibert_stability_path = os.path.join(
    SCIBERT_STEP11_DIR,
    "step11_scibert_directional_stability.csv"
)

scibert_stability_summary_path = os.path.join(
    SCIBERT_STEP11_DIR,
    "step11_scibert_stability_summary.csv"
)

medical_scibert_contrast_path = os.path.join(
    SCIBERT_STEP11_DIR,
    "step11_medical_scibert_all_dimension_contrast.csv"
)

scibert_stability_df.to_csv(scibert_stability_path, index=False, encoding="utf-8-sig")
scibert_stability_summary.to_csv(scibert_stability_summary_path, index=False, encoding="utf-8-sig")
medical_scibert_contrast_df.to_csv(medical_scibert_contrast_path, index=False, encoding="utf-8-sig")

print("\nSaved SciBERT stability outputs:")
print(scibert_stability_path)
print(scibert_stability_summary_path)
print(medical_scibert_contrast_path)

# ============================================================
# 11.6 Final Step 11 evaluation summary
# ============================================================

print("\n========== STEP 11E: FINAL EVALUATION SUMMARY ==========")

final_stability_summary = pd.concat(
    [tfidf_stability_summary, scibert_stability_summary],
    ignore_index=True,
    sort=False
)

display(final_stability_summary)

final_stability_summary_path = os.path.join(
    STEP11_OUTPUT_DIR,
    "step11_final_stability_summary.csv"
)

final_stability_summary.to_csv(
    final_stability_summary_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved final Step 11 outputs:")
print(final_stability_summary_path)

print("\n========== STEP 11 COMPLETE ==========")

In [ ]:
# ============================================================
# STEP 12: Final Analysis
# Cross-domain stability of AI-writing signals
# TF-IDF + UMAP vs SciBERT + UMAP
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

STEP12_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "step12_final_analysis")
os.makedirs(STEP12_OUTPUT_DIR, exist_ok=True)

# ============================================================
# 12.1 Collect final KMeans metrics
# ============================================================

print("========== STEP 12A: FINAL KMEANS COMPARISON ==========")

final_kmeans = kmeans_summary.copy()

display(final_kmeans)

final_kmeans_path = os.path.join(
    STEP12_OUTPUT_DIR,
    "step12_final_kmeans_comparison.csv"
)

final_kmeans.to_csv(final_kmeans_path, index=False, encoding="utf-8-sig")

print("Saved:")
print(final_kmeans_path)

# ============================================================
# 12.2 Collect final stability summaries
# ============================================================

print("\n========== STEP 12B: FINAL STABILITY COMPARISON ==========")

display(final_stability_summary)

final_stability_path = os.path.join(
    STEP12_OUTPUT_DIR,
    "step12_final_stability_comparison.csv"
)

final_stability_summary.to_csv(
    final_stability_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:")
print(final_stability_path)

# ============================================================
# 12.3 Extract important values for final interpretation
# ============================================================

def get_metric(df, representation, domain, algorithm, metric):
    value = df[
        (df["representation"] == representation) &
        (df["domain"] == domain) &
        (df["algorithm"] == algorithm)
    ][metric].values

    if len(value) == 0:
        return np.nan

    return float(value[0])

tfidf_cs_purity = get_metric(final_kmeans, "TF-IDF + UMAP", "Domain A / CS", "KMeans", "purity")
tfidf_cs_ari = get_metric(final_kmeans, "TF-IDF + UMAP", "Domain A / CS", "KMeans", "ari")
tfidf_cs_nmi = get_metric(final_kmeans, "TF-IDF + UMAP", "Domain A / CS", "KMeans", "nmi")

tfidf_med_purity = get_metric(final_kmeans, "TF-IDF + UMAP", "Domain B / Medical", "KMeans", "purity")
tfidf_med_ari = get_metric(final_kmeans, "TF-IDF + UMAP", "Domain B / Medical", "KMeans", "ari")
tfidf_med_nmi = get_metric(final_kmeans, "TF-IDF + UMAP", "Domain B / Medical", "KMeans", "nmi")

scibert_cs_purity = get_metric(final_kmeans, "SciBERT + UMAP", "Domain A / CS", "KMeans", "purity")
scibert_cs_ari = get_metric(final_kmeans, "SciBERT + UMAP", "Domain A / CS", "KMeans", "ari")
scibert_cs_nmi = get_metric(final_kmeans, "SciBERT + UMAP", "Domain A / CS", "KMeans", "nmi")

scibert_med_purity = get_metric(final_kmeans, "SciBERT + UMAP", "Domain B / Medical", "KMeans", "purity")
scibert_med_ari = get_metric(final_kmeans, "SciBERT + UMAP", "Domain B / Medical", "KMeans", "ari")
scibert_med_nmi = get_metric(final_kmeans, "SciBERT + UMAP", "Domain B / Medical", "KMeans", "nmi")

tfidf_presence_jaccard = float(tfidf_stability_summary["presence_jaccard_all_terms"].iloc[0])
tfidf_overall_direction = float(tfidf_stability_summary["overall_direction_agreement"].iloc[0])
tfidf_ai_direction = float(tfidf_stability_summary["ai_vocab_direction_agreement"].iloc[0])
tfidf_human_direction = float(tfidf_stability_summary["human_vocab_direction_agreement"].iloc[0])

scibert_overall_direction = float(scibert_stability_summary["overall_direction_agreement"].iloc[0])
scibert_ai_direction = float(scibert_stability_summary["ai_dimension_direction_agreement"].iloc[0])
scibert_human_direction = float(scibert_stability_summary["human_dimension_direction_agreement"].iloc[0])
scibert_ai_jaccard = float(scibert_stability_summary["dimension_jaccard_ai"].iloc[0])
scibert_human_jaccard = float(scibert_stability_summary["dimension_jaccard_human"].iloc[0])

# ============================================================
# 12.4 Create final interpretation table
# ============================================================

final_interpretation_table = pd.DataFrame([
    {
        "method": "TF-IDF + UMAP",
        "source_domain_cs_purity": tfidf_cs_purity,
        "source_domain_cs_ari": tfidf_cs_ari,
        "source_domain_cs_nmi": tfidf_cs_nmi,
        "target_domain_medical_purity": tfidf_med_purity,
        "target_domain_medical_ari": tfidf_med_ari,
        "target_domain_medical_nmi": tfidf_med_nmi,
        "cross_domain_stability": tfidf_overall_direction,
        "main_interpretation": "TF-IDF vocabulary transfers well, but UMAP-reduced TF-IDF clustering is weak."
    },
    {
        "method": "SciBERT + UMAP",
        "source_domain_cs_purity": scibert_cs_purity,
        "source_domain_cs_ari": scibert_cs_ari,
        "source_domain_cs_nmi": scibert_cs_nmi,
        "target_domain_medical_purity": scibert_med_purity,
        "target_domain_medical_ari": scibert_med_ari,
        "target_domain_medical_nmi": scibert_med_nmi,
        "cross_domain_stability": scibert_overall_direction,
        "main_interpretation": "SciBERT + UMAP gives strong source-domain separation and strong cross-domain transfer."
    }
])

print("\n========== STEP 12C: FINAL INTERPRETATION TABLE ==========")
display(final_interpretation_table)

final_interpretation_path = os.path.join(
    STEP12_OUTPUT_DIR,
    "step12_final_interpretation_table.csv"
)

final_interpretation_table.to_csv(
    final_interpretation_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:")
print(final_interpretation_path)

# ============================================================
# 12.5 Plot final KMeans comparison
# ============================================================

print("\n========== STEP 12D: FINAL METRIC PLOTS ==========")

plot_df = final_kmeans[
    final_kmeans["algorithm"] == "KMeans"
].copy()

plot_df["method_domain"] = (
    plot_df["representation"] + "\n" + plot_df["domain"]
)

for metric in ["purity", "ari", "nmi"]:
    plt.figure(figsize=(10, 5))
    plt.bar(plot_df["method_domain"], plot_df[metric])
    plt.title(f"Final KMeans {metric.upper()} Comparison")
    plt.ylabel(metric.upper())
    plt.xticks(rotation=30, ha="right")
    plt.ylim(0, 1)
    plt.tight_layout()

    plot_path = os.path.join(
        STEP12_OUTPUT_DIR,
        f"step12_final_kmeans_{metric}_comparison.png"
    )

    plt.savefig(plot_path, dpi=300)
    plt.show()

    print("Saved plot:", plot_path)

# ============================================================
# 12.6 Write final report text
# ============================================================

final_report = f"""
Cross-Domain AI vs Human Abstract Analysis
Final Step 12 Analysis
============================================================

Pipeline Summary
------------------------------------------------------------
The experiment followed the same cross-domain pipeline:

Step 1: Corpus Collection
- CS and Medical abstracts were collected in AI vs Human format.

Step 2: Preprocessing
- Text was cleaned, tokenized, lemmatized for TF-IDF, and checked for prompt leakage.
- SciBERT used cleaned natural text.

Step 3: Source-domain feature learning
- TF-IDF was fitted only on Domain A / CS.
- SciBERT embeddings were extracted for Domain A / CS.
- No Medical data was used during source-domain feature learning.

Step 4: Contrast Computation
- TF-IDF contrast was computed as mean(TF-IDF_AI) - mean(TF-IDF_Human).
- SciBERT contrast was computed as mean(SciBERT_AI) - mean(SciBERT_Human).

Step 5: Contrast Feature Selection
- TF-IDF selected 100 AI-dominant words and 100 Human-dominant words.
- SciBERT selected 100 AI-dominant embedding dimensions and 100 Human-dominant embedding dimensions.

Step 6: Domain A Matrix Construction
- CS TF-IDF DTM shape: {X_cs_tfidf_dtm.shape}
- CS SciBERT selected matrix shape: {X_cs_scibert_selected.shape}

Step 7: Domain A UMAP + Clustering
- UMAP was applied before clustering.
- KMeans, Agglomerative Clustering, and GMM were tested.

Step 8: Cross-Domain Transfer
- The same CS-fitted TF-IDF vectorizer was applied to Medical using transform().
- The same CS-selected TF-IDF vocabulary was used for Medical.
- The same CS-selected SciBERT dimensions were used for Medical.
- The same CS-fitted SciBERT scaler was used for Medical.

Step 9: Domain B Matrix Construction
- Medical TF-IDF DTM shape: {X_medical_tfidf_dtm.shape}
- Medical SciBERT selected matrix shape: {X_medical_scibert_selected.shape}
- Medical TF-IDF all-zero rows under selected vocabulary: {int((np.diff(X_medical_tfidf_dtm.indptr) == 0).sum())}

Step 10: Domain B UMAP + Clustering
- The CS-fitted UMAP reducers were used to transform Medical.
- UMAP was not refitted on Medical.

Step 11: Evaluation
- Purity, ARI, NMI, Jaccard, directional stability, and coherence were evaluated.

Final KMeans Results
------------------------------------------------------------

TF-IDF + UMAP on CS:
- Purity: {tfidf_cs_purity:.6f}
- ARI: {tfidf_cs_ari:.6f}
- NMI: {tfidf_cs_nmi:.6f}

TF-IDF + UMAP on Medical:
- Purity: {tfidf_med_purity:.6f}
- ARI: {tfidf_med_ari:.6f}
- NMI: {tfidf_med_nmi:.6f}

SciBERT + UMAP on CS:
- Purity: {scibert_cs_purity:.6f}
- ARI: {scibert_cs_ari:.6f}
- NMI: {scibert_cs_nmi:.6f}

SciBERT + UMAP on Medical:
- Purity: {scibert_med_purity:.6f}
- ARI: {scibert_med_ari:.6f}
- NMI: {scibert_med_nmi:.6f}

TF-IDF Stability Results
------------------------------------------------------------
- Presence Jaccard for selected terms: {tfidf_presence_jaccard:.6f}
- Overall direction agreement: {tfidf_overall_direction:.6f}
- AI vocabulary direction agreement: {tfidf_ai_direction:.6f}
- Human vocabulary direction agreement: {tfidf_human_direction:.6f}

Interpretation:
The selected TF-IDF vocabulary transfers strongly to the Medical domain. Most selected CS contrast terms are present in Medical, and the AI-dominant vocabulary is especially stable. However, after UMAP reduction, TF-IDF clustering performance becomes weak. This suggests that sparse lexical TF-IDF features may lose useful AI/Human separation when compressed by UMAP.

SciBERT Stability Results
------------------------------------------------------------
- AI dimension Jaccard: {scibert_ai_jaccard:.6f}
- Human dimension Jaccard: {scibert_human_jaccard:.6f}
- Overall direction agreement: {scibert_overall_direction:.6f}
- AI dimension direction agreement: {scibert_ai_direction:.6f}
- Human dimension direction agreement: {scibert_human_direction:.6f}

Interpretation:
SciBERT embedding dimensions show strong directional stability across CS and Medical. Although exact top-dimension Jaccard overlap is moderate, the selected CS-derived SciBERT dimensions mostly preserve their AI/Human direction in the Medical domain.

Final Conclusion
------------------------------------------------------------
The UMAP-augmented cross-domain experiment shows that SciBERT is more effective than TF-IDF for capturing stable AI-writing signals across domains.

TF-IDF + UMAP:
- TF-IDF vocabulary stability is strong.
- However, TF-IDF + UMAP clustering is weak in both CS and Medical.
- Therefore, TF-IDF + UMAP should be treated as a weaker baseline or ablation.

SciBERT + UMAP:
- SciBERT + UMAP achieves excellent source-domain separation in CS.
- It also transfers strongly to Medical.
- Medical KMeans results remain high: Purity = {scibert_med_purity:.6f}, ARI = {scibert_med_ari:.6f}, NMI = {scibert_med_nmi:.6f}.
- Therefore, SciBERT + UMAP provides stronger evidence of cross-domain stability of AI-writing signals.

Main Finding:
SciBERT embeddings combined with UMAP provide a more robust cross-domain representation of AI-vs-Human abstract writing signals than TF-IDF features combined with UMAP.
"""

final_report_path = os.path.join(
    STEP12_OUTPUT_DIR,
    "step12_final_cross_domain_analysis_report.txt"
)

with open(final_report_path, "w", encoding="utf-8") as f:
    f.write(final_report)

print("\n========== STEP 12E: FINAL REPORT ==========")
print(final_report)

print("\nSaved final report:")
print(final_report_path)

# ============================================================
# 12.7 Save all important final artifacts together
# ============================================================

final_artifacts_summary = pd.DataFrame([
    {"artifact": "Final KMeans comparison", "path": final_kmeans_path},
    {"artifact": "Final stability comparison", "path": final_stability_path},
    {"artifact": "Final interpretation table", "path": final_interpretation_path},
    {"artifact": "Final text report", "path": final_report_path},
    {"artifact": "All clustering metrics", "path": all_clustering_metrics_path},
    {"artifact": "TF-IDF stability details", "path": tfidf_stability_path},
    {"artifact": "TF-IDF NPMI coherence", "path": tfidf_coherence_path},
    {"artifact": "SciBERT stability details", "path": scibert_stability_path},
])

final_artifacts_path = os.path.join(
    STEP12_OUTPUT_DIR,
    "step12_final_artifacts_summary.csv"
)

final_artifacts_summary.to_csv(
    final_artifacts_path,
    index=False,
    encoding="utf-8-sig"
)

print("\n========== STEP 12F: FINAL ARTIFACTS ==========")
display(final_artifacts_summary)

print("\nSaved final artifacts summary:")
print(final_artifacts_path)

print("\n========== STEP 12 COMPLETE ==========")